# Hybrid Search for RAG: Answer Company Questions Without Retraining

> **The story:** Riverside's fine-tuned model can learn stable behavior such as house style and instruction following. Company knowledge is different: policies change, answers need citations, and access may depend on the employee. Retrieval-augmented generation keeps that knowledge in documents and supplies the current approved passages at request time.
>
> **Where you are:** Notebooks 01-03 changed how the assistant behaves. This notebook asks how it should answer questions such as `Can I upload a draft manuscript to a public AI service?`, `What does RIGHTS-17 require?`, or `How much parental leave is available?` without baking those answers into model weights.
>
> **Notation:** $q$ is a query; $d$ is a policy passage; $r_s(d)$ is its rank under retriever $s$; $k$ is the RRF damping constant; $K$ is the cutoff in Recall@$K$.

## 0 · The Challenge

> **The mission**: answer private Riverside policy questions with current, citable passages without retraining whenever HR, Legal, Security, or Finance publishes a revision.

**Why not fine-tune the policies into the model?**

- A policy can change tomorrow while trained weights remain stale.
- Employees need the exact approved wording and source, not plausible recall.
- Access controls may allow one employee to see a document that another cannot.
- Deleted or superseded text must disappear from answers immediately.
- Re-indexing documents is faster and cheaper than retraining and re-evaluating a model.

**What's blocking us:**
Dense retrieval understands paraphrases but may bury exact policy codes such as `RIGHTS-17`. BM25 finds exact codes but cannot understand a question such as `What support is available when a baby arrives?` when the document says `parental leave`.

**What this chapter unlocks:**
A local hybrid retriever over Riverside's company policies. Dense and lexical search will each fail on a different real query; their complementary rankings create the need for fusion.

| Act | Riverside question | Result |
| --- | --- | --- |
| 1 | Why retrieve policies instead of training them into weights? | RAG versus fine-tuning decision |
| 2 | Can dense search preserve exact policy identifiers? | Exact-code failure |
| 3 | Can BM25 understand employee paraphrases? | Vocabulary-mismatch failure |
| 4 | How can incompatible rankings be combined? | RRF and normalized weighted fusion |
| 5 | Did retrieval improve on labeled questions? | Recall@$K$, MRR, and failure review |
| 6 | What should Riverside operate? | Versioned local policy indexes feeding cited context to the assistant |

## Architecture Decision: RAG or Fine-Tuning?

The choice depends on **what must change**.

| Need | Fine-tuning | RAG |
| --- | --- | --- |
| Persistent response style or format | Strong fit: behavior is practiced repeatedly | Retrieved instructions can help, but consume context every request |
| Current HR, copyright, security, or finance policy | Poor fit: facts become stale and have weak provenance | Strong fit: update the document/index and return the source passage |
| Employee-specific authorization | Weights cannot reliably enforce document-level access | Filter before retrieval and preserve ACL metadata |
| Immediate correction or deletion | Requires retraining and re-evaluation | Remove or supersede the document and republish the index |
| Citation and audit trail | Weight recall cannot identify the authoritative source | Stable passage IDs and versions travel with the answer |

The practical rule is:

> Fine-tune behavior that should persist. Retrieve knowledge that must remain current, inspectable, authorized, and replaceable.

RAG is not automatically correct. Retrieval can miss the right policy, and generation can still misuse a retrieved passage. This notebook measures the retrieval layer; the next RAG evaluation chapter checks whether the final answer is grounded and correct.

| Layer | Riverside checks |
| --- | --- |
| Fine-tuned decoder | Does the answer follow Riverside's response contract? |
| Hybrid retrieval | Did the right approved policy passages reach the prompt? |
| RAG evaluation | Is the answer supported and correct? |
| Gateway | Are authorization, latency, fallback, and logging reliable? |

## Before Hybrid Search: Make One Retriever Fail

![Hybrid retrieval storyboard showing exact identifier matching, semantic matching, and reciprocal rank fusion](images/hybrid-retrieval-storyboard.png)

Retrieval is justified, but **hybrid** retrieval is not yet justified.

Start with dense search. It can connect an employee's everyday phrasing with formal policy language, but an unfamiliar identifier such as `RIGHTS-17` may not carry useful semantic meaning.

BM25 fixes that exact-code failure. Then ask for `support when welcoming a baby`; BM25 cannot infer that the relevant document is the parental-leave policy when those words do not overlap.

Only after both methods fail differently do we need a fusion rule such as RRF.

### Optional Reference: Retrieval Topics Beyond the Main Path

The practical path is dense failure, lexical failure, fusion, and evaluation. Continue directly to Part 1 unless you need the scope ledger.

| Coverage | Topics |
| --- | --- |
| Built and measured | Dense retrieval, BM25, RRF, normalized weighted fusion, alpha tuning, Recall@$K$, MRR, two-stage retrieval, cross-encoder reranking |
| Explained but not fully built | Approximate vector indexing, query rewriting, nDCG |
| Named but out of scope | Learned fusion, domain-specific embedding selection, chunking experiments, metadata filtering |

The closing ledger explains why each deferred topic is outside this notebook. Naming the landscape here should not interrupt the failure-driven route through the techniques that are actually executed.

## Learning Route

1. Establish why current corpus facts belong in retrieval rather than model weights.
2. Make dense retrieval fail on an exact rare term.
3. Make BM25 fail on a paraphrase.
4. Fuse complementary ranked lists with RRF; compare normalized weighted fusion.
5. Tune and evaluate with Recall@$K$ and MRR.
6. Translate the measured result into Riverside's RAG architecture.

Advanced indexing, query rewriting, and packaging sections are optional references. The practical path is the six-step failure chain above.

In [ ]:
# Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("sentence_transformers", "sentence-transformers"),
    ("langchain", "langchain"),
    ("langchain_community", "langchain-community"),
    ("rank_bm25", "rank-bm25"),
]

# Only install packages that aren't already importable, to keep re-runs fast
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  [OK]  {pkg}")
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  [OK]  {pkg} installed")

print("\nDependencies ready.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
import re

warnings.filterwarnings("ignore")

# Set consistent plot styling for every chart in this notebook
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)  # fix the seed so quoted numbers reproduce on re-run

print("Libraries loaded successfully.")


```mermaid
flowchart LR
    A["Semantic search\n(dense cosine)"] --> B["Failure probe:\nexact code\nRIGHTS-17"]
    C["BM25\n(lexical)"] --> D["Failure probe:\nemployee paraphrase\nwelcoming a baby"]
    B --> E["Fix: hybrid\ncombines both\nsignal types"]
    D --> E
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

---

## Part 1 - Can Retrieval Supply the Current Policy?

The first component must pass a simple test:

> Given an employee question, does the retriever place the authoritative approved passage near the top before generation begins?

Use ten small Riverside documents covering copyright, remote work, public AI use, expenses, information security, parental leave, procurement, editorial style, conduct, and records retention.

The corpus is intentionally inspectable. Exact codes behave like legal or finance identifiers; plain-language employee questions expose vocabulary mismatch.

In [ ]:
# Riverside company-policy corpus used throughout the notebook.
documents = [
    "Riverside Copyright and Licensing Policy RIGHTS-17: Reusing unpublished manuscript text outside Riverside requires written Legal approval and confirmation from the rights owner.",
    "Riverside Remote Work Policy: Eligible employees may work away from the office up to three days each week with manager approval and must attend core collaboration hours.",
    "Riverside AI Usage Policy: Unpublished manuscripts, employee records, and confidential business data must never be uploaded to public generative AI services. Use approved local systems only.",
    "Riverside Travel and Expenses Policy: Economy airfare is required for flights under six hours. Receipts are required for expenses above twenty-five dollars and claims must be submitted within thirty days.",
    "Riverside Information Security Policy: Confidential drafts must be encrypted in transit and at rest. Access follows least privilege and is removed when no longer needed.",
    "Riverside Parental Leave Policy: Eligible employees receive sixteen weeks of paid leave following birth, adoption, or foster placement.",
    "Riverside Procurement Policy FIN-42: Purchases above five thousand dollars require Finance approval and two supplier quotes before commitment.",
    "Riverside Editorial Style Guide: Narrative defaults to past tense. Present tense requires an explicit exception from the assigned editor.",
    "Riverside Code of Conduct: Employees may report harassment to People Operations or the anonymous ethics line. Retaliation is prohibited.",
    "Riverside Records Retention Policy: Employment records are retained for seven years, signed author contracts permanently, and routine drafts according to the publishing schedule.",
]

# One query fixture drives every experiment, chart, validation split, and production example.
POLICY_QUERY_FIXTURE = {
    "rights_code": ("RIGHTS-17", [0]),
    "remote_work": ("work from home schedule", [1]),
    "public_ai": ("can staff paste draft novels into ChatGPT", [2]),
    "procurement": ("purchase needs two vendor bids", [6]),
    "parental_leave": ("support when welcoming a baby", [5]),
    "records_retention": ("how long are personnel files kept", [9]),
    "conduct": ("where can I report workplace bullying", [8]),
    "travel_receipts": ("when must travel receipts be filed", [3]),
    "rights_reuse": ("external manuscript reuse approval", [0]),
    "copyright_owner": ("copyright approval rights owner", [0]),
    "expense_deadline": ("expense receipts deadline", [3]),
    "public_ai_manuscripts": ("public AI use for manuscripts", [2]),
    "supplier_quotes": ("purchase approval supplier quotes", [6]),
}

VALIDATION_QUERY_KEYS = [
    "rights_code",
    "remote_work",
    "public_ai",
    "procurement",
    "parental_leave",
]
HELD_OUT_QUERY_KEYS = ["records_retention", "conduct", "travel_receipts"]
validation_queries = [POLICY_QUERY_FIXTURE[key] for key in VALIDATION_QUERY_KEYS]
held_out_queries = [POLICY_QUERY_FIXTURE[key] for key in HELD_OUT_QUERY_KEYS]

print(f"Dataset: {len(documents)} Riverside policy passages\n")
for index, document in enumerate(documents, 1):
    print(f"{index:2d}. {document}")


### Failure Mode 1: Dense Search Can Lose an Exact Policy Code

**Query:** `RIGHTS-17`

A policy code is highly informative to Riverside's systems but carries little ordinary-language meaning for a general embedding model. Dense search may place a thematically related policy above the exact coded document.

This is the failure lexical retrieval is built to prevent: when the user supplies an exact identifier, preserve it.

#### Predict first: what does `RIGHTS-17` mean to an embedding model?

- **Document 1:** Copyright and Licensing Policy, containing the exact code.
- **Document 7:** Procurement Policy, another document with approval language and a policy code.
- **Another policy:** the model treats the unfamiliar token as weak noise.

Commit to a prediction, then inspect the actual ranking.

> **PyTorch → Keras:** `SentenceTransformer("all-MiniLM-L6-v2")` loads a pretrained sentence-embedding model built on a PyTorch `nn.Module` transformer backbone; `.encode(...)` runs a batched, no-grad forward pass and returns dense NumPy vectors for each input string. **Keras/TF equivalent:** the closest TF-native pattern is a TF-Hub/Keras-Hub sentence encoder (e.g. `hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4")`) called on a batch of strings, or a `TFAutoModel` from `transformers` with manual mean-pooling of its last hidden state — Keras has no single drop-in `.encode()` convenience method like `sentence-transformers`.


In [ ]:
# Dense retrieval over the Riverside policy corpus.
print("Loading sentence transformer model...")
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = semantic_model.encode(documents, show_progress_bar=False)
print(f"Encoded {len(documents)} documents into {doc_embeddings.shape[1]} dimensions.\n")


def semantic_search(query, top_k=3):
    """Rank documents by dense cosine similarity."""
    query_embedding = semantic_model.encode([query], show_progress_bar=False)
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    return [(index, similarities[index]) for index in ranked_indices]


query1, [rare_term_target] = POLICY_QUERY_FIXTURE["rights_code"]
results = semantic_search(query1, top_k=5)

print(f"Query: {query1!r}\n")
for rank, (index, score) in enumerate(results, 1):
    marker = "  <-- exact policy" if index == rare_term_target else ""
    print(f"Rank {rank} (score={score:.3f}): Document {index + 1}{marker}")
    print(f"  {documents[index]}\n")

if results[0][0] == rare_term_target:
    print("Dense search found the exact policy on this run.")
    print("That does not make identifiers semantic; BM25 still scores the exact code directly.")
else:
    print("Dense search missed the exact coded policy at rank 1.")
    print("The unfamiliar identifier carried less signal than general policy language.")


### Failure Mode 2: BM25 Cannot Infer a Policy from a Paraphrase

**Query:** `support when welcoming a baby`

The authoritative document says `Parental Leave Policy` and refers to birth, adoption, and foster placement. The employee's wording expresses the same need without using those policy terms.

BM25 counts word overlap. With little or no overlap, it has no basis for connecting the question to parental leave.

#### Predict first: will exact-word search find the meaning?

- **Document 6:** Parental Leave Policy, the semantic answer.
- **A document sharing `support`, `baby`, or `welcoming`:** the lexical favorite if such words appear.
- **An arbitrary zero-score document:** possible when no indexed words overlap.

Run BM25 and inspect what actually happens.

In [ ]:
# BM25 lexical retrieval over the same Riverside policy corpus.
def preprocess_text(text):
    """Lowercase text and remove punctuation before lexical matching."""
    return re.sub(r"[^\w\s]", "", text.lower())


tokenized_docs = [preprocess_text(document).split() for document in documents]
bm25 = BM25Okapi(tokenized_docs)


def lexical_search(query, top_k=3):
    """Rank documents by BM25 token overlap."""
    scores = bm25.get_scores(preprocess_text(query).split())
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(index, scores[index]) for index in ranked_indices]


query2, [paraphrase_target] = POLICY_QUERY_FIXTURE["parental_leave"]
results = lexical_search(query2, top_k=5)

print(f"Query: {query2!r}\n")
for rank, (index, score) in enumerate(results, 1):
    marker = "  <-- parental leave policy" if index == paraphrase_target else ""
    print(f"Rank {rank} (score={score:.2f}): Document {index + 1}{marker}")
    print(f"  {documents[index]}\n")

if results[0][0] == paraphrase_target:
    print("BM25 found the policy because this corpus happened to share enough query words.")
else:
    print("BM25 missed the parental-leave policy at rank 1.")
    print("The question and policy mean the same thing but use different words.")


### Code Walkthrough: BM25 Lexical Search Implementation

**What just ran — 4 key patterns:**

---

**`preprocess_text(text)` — normalise vocabulary before tokenisation**
BM25 operates on bags of tokens, so `"Receipts,"` and `"receipts"` must collapse to the same token. The function lowercases and strips punctuation with `re.sub(r"[^\w\s]", "", text)`. Without this, case differences and trailing punctuation inflate the vocabulary, causing mismatches when the same word appears in different surface forms across policies and queries.

---

**`BM25Okapi(tokenized_docs)` — pre-compute the index once at build time**
`rank_bm25.BM25Okapi` precomputes the IDF for every vocabulary term and the average document length across the corpus when the object is constructed. At query time, `bm25.get_scores(tokenized_query)` only does fast arithmetic using those cached values — no corpus scan. The `Okapi` variant adds saturation via the k₁ parameter (default 1.5) that caps the benefit of seeing the same term many times in one document, preventing very long repetitive documents from dominating the ranking.

---

**`lexical_search(query, top_k=3)` — score, sort, slice**
`get_scores()` returns one float per document. `np.argsort(scores)[::-1][:top_k]` sorts descending and slices the top k. The function returns `(doc_index, score)` pairs so callers can display scores alongside results — the score comparison against semantic cosine values in Part 4 is only possible because both functions return the same `(idx, score)` tuple format.

---

**Prediction-check print block — closed-loop pedagogy**
After each retrieval experiment, the code reports the measured target rank and explains a miss in corpus terms, such as a parental-leave question sharing no useful tokens with the formal policy. This keeps the prediction loop honest: an unexpected outcome has a named cause rather than a prewritten success claim.

> **Score note:** BM25 scores are unbounded — a top document may score 5.2 while cosine similarity is capped at 1.0. This scale mismatch is exactly why raw scores cannot be summed before normalisation (Part 6).


### Side-by-Side Comparison: The Search Gap

Let's visualize how semantic and lexical search produce **complementary** results on two different queries:


In [ ]:
# Compare both methods on the two canonical Riverside failure probes
from matplotlib.patches import Patch

test_queries = [
    ("RIGHTS-17", 0, "Exact policy-code test"),
    ("support when welcoming a baby", 5, "Employee-paraphrase test"),
]

# Two rows (one per test query) x two columns (semantic vs lexical) side-by-side panel
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Riverside Policy Search: Semantic vs Lexical", fontsize=14, fontweight="bold")

comparison_results = []
for row, (query, target_doc, test_name) in enumerate(test_queries):
    # Semantic search
    sem_results = semantic_search(query, top_k=10)
    sem_scores = [score for _, score in sem_results]
    sem_docs = [f"Policy {idx}" for idx, _ in sem_results]
    sem_colors = [
        "green" if idx == target_doc else "steelblue" for idx, _ in sem_results
    ]

    # Lexical search
    lex_results = lexical_search(query, top_k=10)
    lex_scores = [score for _, score in lex_results]
    lex_docs = [f"Policy {idx}" for idx, _ in lex_results]
    lex_colors = ["green" if idx == target_doc else "coral" for idx, _ in lex_results]

    sem_rank = next(rank for rank, (idx, _) in enumerate(sem_results, 1) if idx == target_doc)
    lex_rank = next(rank for rank, (idx, _) in enumerate(lex_results, 1) if idx == target_doc)
    comparison_results.append((test_name, query, target_doc, sem_rank, lex_rank))

    # Plot semantic — categorical color-coding (target vs. other) needs its own legend
    axes[row, 0].barh(sem_docs, sem_scores, color=sem_colors, alpha=0.7)
    axes[row, 0].set_xlabel("Cosine Similarity")
    axes[row, 0].set_title(f"{test_name}\nSemantic Search: '{query}'")
    axes[row, 0].invert_yaxis()
    axes[row, 0].legend(
        handles=[
            Patch(facecolor="green", alpha=0.7, label=f"Target policy {target_doc}"),
            Patch(facecolor="steelblue", alpha=0.7, label="Other result"),
        ],
        loc="lower right",
        fontsize=8,
    )

    # Plot lexical — same categorical scheme, different "other" color, still needs its own legend
    axes[row, 1].barh(lex_docs, lex_scores, color=lex_colors, alpha=0.7)
    axes[row, 1].set_xlabel("BM25 Score")
    axes[row, 1].set_title(f"{test_name}\nLexical Search (BM25): '{query}'")
    axes[row, 1].invert_yaxis()
    axes[row, 1].legend(
        handles=[
            Patch(facecolor="green", alpha=0.7, label=f"Target policy {target_doc}"),
            Patch(facecolor="coral", alpha=0.7, label="Other result"),
        ],
        loc="lower right",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

print("\nMeasured target ranks (green bars):")
for test_name, query, target_doc, sem_rank, lex_rank in comparison_results:
    print(
        f"- {test_name}: '{query}' -> policy {target_doc}; "
        f"semantic rank={sem_rank}, lexical rank={lex_rank}"
    )

exact_result = comparison_results[0]
paraphrase_result = comparison_results[1]
if exact_result[3] > exact_result[4]:
    print("\nRIGHTS-17 favored lexical search on this run, as an exact code often should.")
else:
    print("\nRIGHTS-17 did not produce a lexical win on this run; keep the measured ranks above.")

if paraphrase_result[3] < paraphrase_result[4]:
    print("The parental-leave paraphrase favored semantic search on this run.")
else:
    print("The parental-leave paraphrase did not produce a semantic win on this run.")
print("The methods remain complementary, but this toy corpus does not guarantee either outcome.")


### Common Pitfalls: Trusting One Search Method Alone

**Pitfall #1: Assuming semantic search preserves exact internal identifiers**

**Bad:** Ship semantic-only search for policies where codes such as `RIGHTS-17` and `FIN-42` must be matched exactly.
**Good:** Pair semantic search with a lexical/BM25 pass whenever the corpus contains rare, exact-match-critical identifiers.

**Why it happens:** Dense embeddings represent contextual meaning. An unfamiliar code may have no useful semantic direction, so general words such as `approval` or `policy` can dominate its vector.

**Pitfall #2: Assuming BM25 understands employee paraphrases**

**Bad:** Ship lexical-only search and expect `support when welcoming a baby` to find a document written in terms of parental leave, birth, adoption, and foster placement.
**Good:** Pair lexical search with semantic search whenever employees may describe a policy in everyday language.

**Why it happens:** BM25 only counts token overlap; it has no built-in notion that two different strings can express the same intent.

**Quick health check:** run one exact-identifier query and one realistic employee paraphrase. Record both target ranks for each retriever. If either single method misses a target, the measured gap motivates hybrid retrieval; if neither misses on this small corpus, keep the methods as robustness candidates and test harder queries.


In [ ]:
# Quick health check: exact-code recall vs employee-paraphrase recall.
rare_term_query, [rare_term_target] = POLICY_QUERY_FIXTURE["rights_code"]
paraphrase_query, [paraphrase_target] = POLICY_QUERY_FIXTURE["parental_leave"]

sem_rare_top1 = semantic_search(rare_term_query, top_k=1)[0][0]
lex_rare_top1 = lexical_search(rare_term_query, top_k=1)[0][0]

sem_paraphrase_top1 = semantic_search(paraphrase_query, top_k=1)[0][0]
lex_paraphrase_top1 = lexical_search(paraphrase_query, top_k=1)[0][0]

print(f"Health Check 1 — exact code ({rare_term_query!r}):")
print(
    f"  Semantic top-1: Policy {sem_rare_top1}  "
    f"{'[PASS]' if sem_rare_top1 == rare_term_target else '[MISS]'}"
)
print(
    f"  Lexical  top-1: Policy {lex_rare_top1}  "
    f"{'[PASS]' if lex_rare_top1 == rare_term_target else '[MISS]'}"
)

print(f"\nHealth Check 2 — employee paraphrase ({paraphrase_query!r}):")
print(
    f"  Semantic top-1: Policy {sem_paraphrase_top1}  "
    f"{'[PASS]' if sem_paraphrase_top1 == paraphrase_target else '[MISS]'}"
)
print(
    f"  Lexical  top-1: Policy {lex_paraphrase_top1}  "
    f"{'[PASS]' if lex_paraphrase_top1 == paraphrase_target else '[MISS]'}"
)

semantic_has_blind_spot = (
    sem_rare_top1 != rare_term_target or sem_paraphrase_top1 != paraphrase_target
)
lexical_has_blind_spot = (
    lex_rare_top1 != rare_term_target or lex_paraphrase_top1 != paraphrase_target
)

print("\nMeasured conclusion:")
if semantic_has_blind_spot and lexical_has_blind_spot:
    print("  Each single retriever missed at least one canonical target at rank 1.")
elif semantic_has_blind_spot:
    print("  Only semantic search exposed a top-1 blind spot on these two probes.")
elif lexical_has_blind_spot:
    print("  Only lexical search exposed a top-1 blind spot on these two probes.")
else:
    print("  Neither method missed at rank 1 on this run; use harder labels before claiming a win.")


---

## Part 2 — Semantic Search: Dense Vector Intuition

**Riverside's question for this section:** can dense embeddings connect an employee's everyday wording with the formal language in an approved policy?

Semantic search maps text into a high-dimensional continuous space where semantically similar policies and queries can cluster even when they do not share exact words.

### How It Works

1. **Embedding model**: a neural network (for example, BERT or sentence-transformers) encodes text into a dense vector such as 384 or 768 dimensions.
2. **Similarity metric**: cosine similarity, $\text{similarity}(q, d) = \cos(\theta)$, measures the angle between query and document vectors.
3. **Ranking**: policies are ranked by similarity score (range: -1 to 1, commonly concentrated in a narrower positive band).

The key intuition is that texts with related intent can point in similar directions despite different vocabulary. `work from home schedule` can land near a policy that says employees may `work away from the office`, and `support when welcoming a baby` can land near language about leave following birth or adoption. The smaller the vector angle, the higher the cosine similarity.

### Strengths and Weaknesses

**Strengths:**

- Connects formal policy wording with employee paraphrases.
- Handles synonyms and intent-level similarity.
- Can support multilingual search when a multilingual encoder is selected.

**Weaknesses:**

- Exact identifiers can be diluted by surrounding language.
- A topically similar but wrong policy can rank highly.
- Quality depends on the embedding model and Riverside's labeled query set.

Let's visualize the embedding space:


> **PyTorch → Keras:** `semantic_model.encode(test_queries_viz, show_progress_bar=False)` reuses the same PyTorch sentence-transformer to embed a fresh batch of query strings into 384-dim vectors, which are then projected to 2D with `PCA` purely for visualization. **Keras/TF equivalent:** you'd call an equivalent TF-Hub/`TFAutoModel`-based encoder from a Keras workflow on this same batch of strings and feed its output vectors into the identical `sklearn.decomposition.PCA` call — the embedding call is the only PyTorch-specific step; PCA and plotting are framework-agnostic.


In [ ]:
# Semantic embedding-space visualization.
from sklearn.decomposition import PCA

# Reduce document embeddings to 2D for visualization.
pca = PCA(n_components=2)
doc_embeddings_2d = pca.fit_transform(doc_embeddings)

# Encode policy queries from the shared fixture.
visualization_query_keys = ["remote_work", "public_ai", "procurement"]
test_queries_viz = [POLICY_QUERY_FIXTURE[key][0] for key in visualization_query_keys]
query_embeddings = semantic_model.encode(test_queries_viz, show_progress_bar=False)

# Reuse the same fitted PCA projection so queries land in the documents' 2D space.
query_embeddings_2d = pca.transform(query_embeddings)

plt.figure(figsize=(12, 8))
plt.scatter(
    doc_embeddings_2d[:, 0],
    doc_embeddings_2d[:, 1],
    s=100,
    alpha=0.6,
    c="steelblue",
    label="Policy passages",
)

for index, (x_coord, y_coord) in enumerate(doc_embeddings_2d):
    plt.annotate(
        f"Policy {index}",
        (x_coord, y_coord),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
    )

plt.scatter(
    query_embeddings_2d[:, 0],
    query_embeddings_2d[:, 1],
    s=200,
    alpha=0.8,
    c="red",
    marker="*",
    label="Employee queries",
)

for index, (x_coord, y_coord) in enumerate(query_embeddings_2d):
    plt.annotate(
        test_queries_viz[index],
        (x_coord, y_coord),
        xytext=(5, -15),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color="darkred",
    )

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title(
    "Riverside Policy Embeddings (384D → 2D via PCA)\nEmployee queries projected into the same space"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nPCA is a two-dimensional diagnostic, not proof of retrieval quality.")
for key in visualization_query_keys:
    query, targets = POLICY_QUERY_FIXTURE[key]
    print(f"- {query!r} has labeled target policy {targets[0]}.")
print("Use the ranked retrieval metrics later in the notebook to judge whether those targets were found.")


---

## Part 3 — BM25: Lexical Search with IDF Weighting

**Riverside's question for this section:** can keyword search preserve an exact policy or finance identifier that a general embedding model may not understand?

BM25 scores a document by summing over the query terms, giving each term a weight based on how rare it is across the corpus and how often it appears in this specific document — with diminishing returns as frequency grows.

The full formula is: $\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot |D|/\text{avgdl})}$

Three intuitions are embedded in this single equation:

**IDF gives rare terms more power.** A term that appears in only one document out of ten tells you more than one that appears in all ten. This is why `RIGHTS-17` and `FIN-42` can be strong lexical signals while a common word such as `policy` contributes little.

**Repetition has diminishing returns (TF saturation).** Mentioning `approval` ten times does not make a policy ten times more relevant than mentioning it once. The denominator grows sub-linearly with term frequency, preventing a verbose policy from dominating merely through repetition.

**Long documents do not get a free pass (length normalization).** A long handbook section that mentions `receipts` once is less focused on the query than a short expenses rule centered on receipt deadlines. The length-normalization term adjusts for that difference. `b=0.75` is a common default; `b=0` disables the penalty.

### Strengths and Weaknesses

**Strengths:**

- Exact matching for policy codes and named controls.
- Rare-term prioritization via IDF.
- Fast and interpretable scoring.

**Weaknesses:**

- No understanding of synonyms or intent.
- Vocabulary mismatch between employees and formal policy text.
- Weak results for paraphrased questions with no useful overlap.

Let's implement BM25 from scratch and compare it to sklearn's TF-IDF:


In [ ]:
# Manual BM25 implementation.


def compute_bm25_manual(query, documents, k1=1.5, b=0.75):
    """Manual BM25 implementation with step-by-step calculations."""
    tokenized_docs = [preprocess_text(doc).split() for doc in documents]
    tokenized_query = preprocess_text(query).split()

    document_count = len(documents)
    avgdl = np.mean([len(doc) for doc in tokenized_docs])

    idf_scores = {}
    for term in tokenized_query:
        matching_documents = sum(1 for doc in tokenized_docs if term in doc)
        idf = np.log(
            (document_count - matching_documents + 0.5) / (matching_documents + 0.5) + 1
        )
        idf_scores[term] = idf

    scores = []
    for doc_tokens in tokenized_docs:
        doc_len = len(doc_tokens)
        score = 0.0

        for term in tokenized_query:
            term_frequency = doc_tokens.count(term)
            numerator = term_frequency * (k1 + 1)
            denominator = term_frequency + k1 * (1 - b + b * (doc_len / avgdl))
            score += idf_scores[term] * (numerator / denominator)

        scores.append(score)

    return scores, idf_scores, avgdl


query, [manual_bm25_target] = POLICY_QUERY_FIXTURE["supplier_quotes"]
scores, idf_scores, avgdl = compute_bm25_manual(query, documents)

print(f"Query: {query!r}\n")
print("IDF scores (higher means rarer in this corpus):")
for term, idf in idf_scores.items():
    print(f"  {term!r}: {idf:.3f}")

print(f"\nAverage document length: {avgdl:.1f} words\n")
print("BM25 scores:")

ranked_indices = np.argsort(scores)[::-1][:5]
for rank, index in enumerate(ranked_indices, 1):
    marker = "  <-- procurement target" if index == manual_bm25_target else ""
    print(f"Rank {rank} (score={scores[index]:.2f}): Policy {index}{marker}")
    print(f"  {documents[index][:100]}...\n")

sorted_by_idf = sorted(idf_scores.items(), key=lambda item: item[1], reverse=True)
rarest_term, rarest_idf = sorted_by_idf[0]
print(
    f"The highest-IDF query term is {rarest_term!r} ({rarest_idf:.3f}) for this corpus."
)
print("IDF reflects corpus frequency, not a word's importance in the abstract.")


### BM25 vs TF-IDF: What's the Difference?

TF-IDF is simpler but lacks BM25's sophistication:

| Feature                  | TF-IDF                            | BM25                                  |
| ------------------------ | --------------------------------- | ------------------------------------- |
| Term frequency scaling   | Linear (tf × idf)                 | Saturating (diminishing returns)      |
| Document length handling | Optional normalization            | Built-in length penalty (b parameter) |
| Tuning parameters        | None                              | k₁ (saturation), b (length penalty)   |
| Typical use case         | Simple keyword search, clustering | Information retrieval, search engines |

Let's compare them side-by-side:


In [ ]:
# TF-IDF search.
tfidf_vectorizer = TfidfVectorizer()

# Fit the vectorizer's vocabulary/IDF weights on the corpus and encode every document.
tfidf_matrix = tfidf_vectorizer.fit_transform(
    [preprocess_text(doc) for doc in documents]
)


def tfidf_search(query, top_k=5):
    """Rank policy passages by TF-IDF cosine similarity."""
    query_vec = tfidf_vectorizer.transform([preprocess_text(query)])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    return [(index, similarities[index]) for index in ranked_indices]


query, [comparison_target] = POLICY_QUERY_FIXTURE["supplier_quotes"]
tfidf_results = tfidf_search(query, top_k=5)
bm25_results = lexical_search(query, top_k=5)

df_comparison = pd.DataFrame(
    {
        "Rank": range(1, 6),
        "TF-IDF Policy": [index for index, _ in tfidf_results],
        "TF-IDF Score": [f"{score:.3f}" for _, score in tfidf_results],
        "BM25 Policy": [index for index, _ in bm25_results],
        "BM25 Score": [f"{score:.2f}" for _, score in bm25_results],
    }
)

print(f"Query: {query!r}; labeled target policy: {comparison_target}\n")
print(df_comparison.to_string(index=False))

if tfidf_results[0][0] == comparison_target and bm25_results[0][0] == comparison_target:
    print("\nBoth lexical methods ranked the procurement policy first on this run.")
elif tfidf_results[0][0] == comparison_target:
    print("\nOnly TF-IDF ranked the procurement policy first on this run.")
elif bm25_results[0][0] == comparison_target:
    print("\nOnly BM25 ranked the procurement policy first on this run.")
else:
    print("\nNeither lexical method ranked the labeled procurement policy first on this run.")
print("Their term-frequency and document-length treatments can still produce different orders and scores.")


---

## Part 4 — Why Hybrid Search Can Help: Complementary Signals

**Riverside's question for this section:** do both retrievers contribute useful candidates across exact policy codes and employee paraphrases?

Hybrid search combines semantic and lexical retrieval in three steps:

1. **Semantic search** contributes policies related by meaning, even with different terminology.
2. **Lexical search** contributes policies with exact token matches, including codes and named controls.
3. **Fusion** re-ranks the combined candidate pool.

### The Correct Set Relationship

For a query $Q$:

- Let $S$ be the semantic candidate set.
- Let $L$ be the lexical candidate set.
- Let $C = S \cup L$ be the union available to fusion.
- Let $H_K$ be the top-$K$ results after fusion.

Then $H_K \subseteq C$. Fusion can only rank candidates supplied by at least one retriever, and top-$K$ truncation means the final result is not a superset of both input lists.

The useful hypothesis is therefore empirical rather than guaranteed: a fused candidate pool may improve recall because semantic search can contribute paraphrase matches while BM25 can contribute exact-code matches. Poor fusion, narrow candidate cutoffs, or weak retrievers can still bury the relevant policy.

### Overlap Analysis

Let's measure how much the semantic and lexical candidate sets overlap across Riverside policy queries. Set overlap shows whether the retrievers return different candidates; relevance metrics later determine whether those differences help.


### 4a. Qualitative Comparison — Retrieval Method vs. Query Type

Parts 1–3 established two axes: **which retrieval method** Riverside uses and **what kind of query** an employee types. The canonical probes are the exact code `RIGHTS-17` and the paraphrase `support when welcoming a baby`.

Before measuring, write down a hypothesis rather than a promised result:

| Retrieval method ↓ / Query type → | **Exact identifier or named control** | **Employee paraphrase** |
| --- | --- | --- |
| **Semantic (dense)** | Can retrieve a related policy, but an unfamiliar code may carry weak semantic signal. | Often strong because related intent can survive vocabulary changes; topical near-misses remain possible. |
| **Lexical (BM25)** | Often strong because IDF rewards an exact rare token without needing training examples. | Weak when the question and policy share few useful tokens. |
| **Hybrid (RRF)** | Can retain BM25's exact-match candidate while considering the dense ranking. | Can retain the dense candidate while considering any lexical evidence. |

Hybrid adds latency and can still rank the wrong candidate. Its value must be demonstrated query by query: first inspect the candidate overlap, then compare Recall@$K$ and MRR on the fixed Riverside labels.


In [ ]:
# Analyze overlap for Riverside policy queries.
overlap_query_keys = [
    "rights_code",
    "remote_work",
    "public_ai",
    "procurement",
    "parental_leave",
]
test_queries_overlap = [POLICY_QUERY_FIXTURE[key][0] for key in overlap_query_keys]

overlap_stats = []
for query in test_queries_overlap:
    semantic_candidates = {index for index, _ in semantic_search(query, top_k=5)}
    lexical_candidates = {index for index, _ in lexical_search(query, top_k=5)}

    overlap_stats.append(
        {
            "Query": query,
            "Only Semantic": len(semantic_candidates - lexical_candidates),
            "Overlap": len(semantic_candidates & lexical_candidates),
            "Only Lexical": len(lexical_candidates - semantic_candidates),
        }
    )

df_overlap = pd.DataFrame(overlap_stats)

fig, ax = plt.subplots(figsize=(12, 6))
x_positions = np.arange(len(df_overlap))
width = 0.6

ax.bar(
    x_positions,
    df_overlap["Only Semantic"],
    width,
    label="Only Semantic",
    color="steelblue",
    alpha=0.8,
)
ax.bar(
    x_positions,
    df_overlap["Overlap"],
    width,
    bottom=df_overlap["Only Semantic"],
    label="Overlap",
    color="purple",
    alpha=0.8,
)
ax.bar(
    x_positions,
    df_overlap["Only Lexical"],
    width,
    bottom=df_overlap["Only Semantic"] + df_overlap["Overlap"],
    label="Only Lexical",
    color="coral",
    alpha=0.8,
)

ax.set_ylabel("Number of Policies in Each Top-5 Set")
ax.set_title("Riverside Policy Retrieval Overlap\nSemantic vs lexical top-5 candidates")
ax.set_xticks(x_positions)
ax.set_xticklabels(df_overlap["Query"], rotation=15, ha="right")
ax.legend()
ax.set_ylim(0, 10.5)
plt.tight_layout()
plt.show()

print("\nCandidate-set overlap statistics:")
print(df_overlap.to_string(index=False))
print("\nNon-overlap means the retrievers supplied different candidates.")
print("It does not establish relevance; the labeled benchmark later checks whether fusion helped.")


#### What just happened — and what's the problem?

Semantic and lexical search produced two candidate lists that may overlap only partially. That is useful only if fusion ranks the labeled policy well; candidate diversity by itself is not proof of better retrieval.

**The next technical problem:** the two ranked lists use incompatible score scales:

- Semantic scores: cosine similarity, usually bounded between -1 and 1.
- BM25 scores: corpus-dependent and unbounded above.

Simply adding them lets numerical scale, rather than relevance evidence, determine the ranking. A BM25 value of 10 can dominate a cosine value of 0.8 without meaning the lexical match is better.

Two defensible fusion approaches are:

1. **Reciprocal Rank Fusion (RRF):** ignore raw scores and combine ranks.
2. **Score normalization:** map both score sets to comparable scales before blending (Part 6).

The notebook will measure both on the same Riverside validation queries rather than declaring a universal winner.


---

## Part 5 - Fuse Incompatible Scores with RRF

Semantic and BM25 search now return complementary results, but their raw scores have different meanings and scales. Adding 0.8 cosine similarity to a BM25 score of 10 lets BM25 dominate numerically without proving it is more relevant.

A score-free fusion rule should satisfy two needs:

1. reward a document that appears near the top of either list;
2. reward it again when both retrievers rank it well.

Ranks already share a scale, so assign each appearance a contribution that decreases with rank and sum the contributions:

$$
\operatorname{RRF}(d)=\sum_{s\in S}\frac{1}{k+r_s(d)}.
$$

For document $d$, $r_s(d)$ is its rank under retriever $s$. A top-ranked appearance contributes more than a lower one, and appearances in both lists accumulate. Raw cosine and BM25 values never meet.

The damping constant $k$ controls how sharply rank contributions differ. Smaller $k$ emphasizes the very top; larger $k$ flattens the contribution across more positions. The common default of 60 is a starting point to validate, not a universal optimum.

The next cell implements this rule and compares it with normalized weighted fusion and the known-bad raw-score sum.

### Optional Precision: RRF Does Not Average Rank Numbers

“Combine ranks” is useful shorthand, but RRF sums reciprocal contributions; it does not compute the arithmetic mean of rank positions.

That distinction matters because a higher-ranked appearance contributes more, while a document found by both systems receives two contributions. The implementation below is the useful source of truth: for each ranked list, add `1 / (k + rank)` to that document's fused score.

In [ ]:
# Reciprocal Rank Fusion implementation.


def reciprocal_rank_fusion(semantic_results, lexical_results, k=60):
    """Merge two ranked lists using Reciprocal Rank Fusion."""
    rrf_scores = {}

    for rank, (doc_idx, _) in enumerate(semantic_results, start=1):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank)

    for rank, (doc_idx, _) in enumerate(lexical_results, start=1):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank)

    return sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)


query, [rrf_target] = POLICY_QUERY_FIXTURE["copyright_owner"]
sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)
rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)

print(f"Query: {query!r}; labeled target policy: {rrf_target}\n")
print("Reciprocal Rank Fusion results:")
print("-" * 80)

for rank, (doc_idx, rrf_score) in enumerate(rrf_results[:5], 1):
    sem_rank = next(
        (position + 1 for position, (index, _) in enumerate(sem_results) if index == doc_idx),
        None,
    )
    lex_rank = next(
        (position + 1 for position, (index, _) in enumerate(lex_results) if index == doc_idx),
        None,
    )
    marker = "  <-- labeled target" if doc_idx == rrf_target else ""

    print(f"Rank {rank} (RRF={rrf_score:.4f}): Policy {doc_idx}{marker}")
    print(f"  Semantic rank: {sem_rank if sem_rank else 'Not in top-10'}")
    print(f"  Lexical rank:  {lex_rank if lex_rank else 'Not in top-10'}")
    print(f"  {documents[doc_idx][:100]}...\n")

target_rrf_rank = next(
    rank for rank, (doc_idx, _) in enumerate(rrf_results, 1) if doc_idx == rrf_target
)
print(f"The labeled copyright policy finished at RRF rank {target_rrf_rank}.")
print("RRF avoids raw-score scale mismatch, but its relevance quality still depends on the input rankings.")


### RRF vs Weighted Score Fusion

An alternative is **weighted score fusion**, which blends normalized scores using a tunable parameter α: $\text{hybrid\_score}(d) = (1 - \alpha) \cdot \text{norm(lex)} + \alpha \cdot \text{norm(sem)}$

The α parameter is the conceptual core: set it near 0 and lexical dominates; near 1 and semantic dominates. Normalization (e.g., min-max scaling to [0, 1]) is required first so BM25's unbounded values don't overwhelm cosine scores.

**Comparison:**

| Method          | Pros                                      | Cons                                   |
| --------------- | ----------------------------------------- | -------------------------------------- |
| RRF             | Rank-based, robust to outliers, no tuning | Ignores score magnitudes               |
| Weighted Fusion | Leverages score confidence, tunable α     | Requires score normalization, outliers |

Let's compare both on the same query:


In [ ]:
# Score normalization and weighted fusion.


def min_max_normalize(scores):
    """Min-max normalization to [0, 1]."""
    scores = np.array(scores)
    min_score = scores.min()
    max_score = scores.max()
    if max_score == min_score:
        return np.ones_like(scores)
    return (scores - min_score) / (max_score - min_score)


def weighted_score_fusion(semantic_results, lexical_results, alpha=0.5):
    """Merge normalized scores with alpha weight on semantic retrieval."""
    sem_docs = [index for index, _ in semantic_results]
    sem_scores_norm = min_max_normalize([score for _, score in semantic_results])

    lex_docs = [index for index, _ in lexical_results]
    lex_scores_norm = min_max_normalize([score for _, score in lexical_results])

    combined_scores = {}
    for index, norm_score in zip(sem_docs, sem_scores_norm):
        combined_scores[index] = combined_scores.get(index, 0) + alpha * norm_score

    for index, norm_score in zip(lex_docs, lex_scores_norm):
        combined_scores[index] = combined_scores.get(index, 0) + (1 - alpha) * norm_score

    return sorted(combined_scores.items(), key=lambda item: item[1], reverse=True)


query, [fusion_target] = POLICY_QUERY_FIXTURE["rights_reuse"]
sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)

rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
weighted_results = weighted_score_fusion(sem_results, lex_results, alpha=0.5)

df_compare = pd.DataFrame(
    {
        "Rank": range(1, 6),
        "RRF Policy": [index for index, _ in rrf_results[:5]],
        "RRF Score": [f"{score:.4f}" for _, score in rrf_results[:5]],
        "Weighted Policy": [index for index, _ in weighted_results[:5]],
        "Weighted Score": [f"{score:.4f}" for _, score in weighted_results[:5]],
    }
)

print(f"Query: {query!r}; labeled target policy: {fusion_target}\n")
print("RRF vs weighted score fusion (α=0.5):\n")
print(df_compare.to_string(index=False))

rrf_target_rank = next(rank for rank, (index, _) in enumerate(rrf_results, 1) if index == fusion_target)
weighted_target_rank = next(
    rank for rank, (index, _) in enumerate(weighted_results, 1) if index == fusion_target
)
print(
    f"\nMeasured target ranks — RRF: {rrf_target_rank}; "
    f"weighted fusion: {weighted_target_rank}."
)
if rrf_target_rank < weighted_target_rank:
    print("RRF ranked the labeled policy higher on this query.")
elif weighted_target_rank < rrf_target_rank:
    print("Weighted fusion ranked the labeled policy higher on this query.")
else:
    print("The two fusion methods tied on the labeled policy rank for this query.")


### Proof: RRF vs Weighted Fusion on This Dataset

Let's measure which fusion method actually performs better on our validation set using **Recall@5** as the metric.


In [ ]:
# Validation: RRF vs weighted fusion.
validation_queries_rrf = validation_queries


def compute_recall_at_k(results, relevant_docs, k=5):
    """Compute Recall@K: fraction of labeled relevant policies in the top K."""
    retrieved = {index for index, _ in results[:k]}
    relevant = set(relevant_docs)
    if not relevant:
        return 0.0
    return len(retrieved & relevant) / len(relevant)


rrf_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
    rrf_recalls.append(compute_recall_at_k(rrf_results, relevant_docs, k=5))

avg_rrf_recall = np.mean(rrf_recalls)

weighted_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    weighted_results = weighted_score_fusion(sem_results, lex_results, alpha=0.5)
    weighted_recalls.append(compute_recall_at_k(weighted_results, relevant_docs, k=5))

avg_weighted_recall = np.mean(weighted_recalls)
recall_delta = avg_rrf_recall - avg_weighted_recall

print("Fusion Method Comparison on Riverside Validation Queries")
print("=" * 80)
print(f"RRF (k=60):                 Recall@5 = {avg_rrf_recall:.3f}")
print(f"Weighted Fusion (α=0.5):    Recall@5 = {avg_weighted_recall:.3f}")
print(f"Delta (RRF - weighted):     {recall_delta:.3f}")

if recall_delta > 0:
    print("\nRRF achieved higher Recall@5 on this five-query policy set.")
elif recall_delta < 0:
    print("\nWeighted fusion achieved higher Recall@5 on this five-query policy set.")
else:
    print("\nThe two fusion methods tied on Recall@5 for this five-query policy set.")
print("RRF avoids score normalization; weighted fusion exposes α as a parameter to validate.")


In [ ]:
# Your turn — RRF constant k.
# Change k_val (try 10, 60, 200) and observe how rank weighting shifts.
# Smaller k = steeper decay; larger k = gentler decay.

k_val = 60

query_ex, [query_ex_target] = POLICY_QUERY_FIXTURE["rights_reuse"]
sem_ex = semantic_search(query_ex, top_k=10)
lex_ex = lexical_search(query_ex, top_k=10)
rrf_ex = reciprocal_rank_fusion(sem_ex, lex_ex, k=k_val)

print(f"RRF with k={k_val} (query: {query_ex!r}; target policy: {query_ex_target}):")
for rank, (doc_idx, score) in enumerate(rrf_ex[:5], 1):
    marker = "  <-- target" if doc_idx == query_ex_target else ""
    print(f"  Rank {rank}: Policy {doc_idx} (RRF score={score:.4f}){marker}")

query_ex_rank = next(rank for rank, (index, _) in enumerate(rrf_ex, 1) if index == query_ex_target)
print(f"\nAt k={k_val}, the labeled policy ranked {query_ex_rank}.")
print("k=60 is a common baseline, not a guaranteed optimum; compare values on Riverside's labels.")


### Common Pitfalls: Fusing Ranked Lists Naively

**Pitfall #1: Summing raw scores without normalization**

**Bad:** `combined_score = bm25_score + cosine_score`. BM25 is corpus-dependent and unbounded, while cosine similarity is bounded, so the larger numerical scale can dominate without representing stronger relevance.
**Good:** Normalize both score sets before blending, or avoid raw scores and fuse by rank with RRF.

**Pitfall #2: Assuming one fusion rule wins everywhere**

**Bad:** Pick RRF or weighted fusion once and treat it as universally best.
**Good:** Measure both on a labeled policy-query set. RRF removes one tuning problem; weighted fusion may rank a target better when its normalization and α fit the data.

**Quick health check:** compute the unnormalized raw-score sum and compare its Recall@5 with RRF and normalized weighted fusion on the same Riverside labels. Report the measured relationship, including a tie or an unexpected raw-sum win on this small corpus.


In [ ]:
#  Quick Health Check: naive raw-score averaging vs RRF
# Prove Pitfall #1 rather than asserting it: fuse with UN-normalized raw scores
# (no min-max, no z-score) and compare recall against RRF on the same validation set.


def naive_raw_fusion(semantic_results, lexical_results):
    """Sum RAW scores with no normalization — the mistake described above."""
    combined = {}
    for idx, score in semantic_results:
        combined[idx] = combined.get(idx, 0) + score
    for idx, score in lexical_results:
        combined[idx] = combined.get(idx, 0) + score
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


# Measure recall for the naive (un-normalized) fusion on the same validation queries
naive_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    naive_results = naive_raw_fusion(sem_results, lex_results)
    naive_recalls.append(compute_recall_at_k(naive_results, relevant_docs, k=5))

avg_naive_recall = np.mean(naive_recalls)

print("Naive raw-score fusion (no normalization) vs RRF vs Weighted Fusion")
print("=" * 80)
print(f"Naive raw sum (BUG):        Recall@5 = {avg_naive_recall:.3f}")
print(f"RRF (k=60):                 Recall@5 = {avg_rrf_recall:.3f}")
print(f"Weighted Fusion (α=0.5):    Recall@5 = {avg_weighted_recall:.3f}")

# Branch the interpretation on what the numbers actually showed on this dataset
if avg_naive_recall < min(avg_rrf_recall, avg_weighted_recall):
    print("\n-> Confirmed: un-normalized raw-score fusion measurably underperforms")
    print("   both RRF and normalized weighted fusion on this dataset — BM25's")
    print(
        "   unbounded scores really do drown out cosine similarity when summed directly."
    )
elif avg_naive_recall == min(avg_rrf_recall, avg_weighted_recall):
    print("\n-> On this small validation set, naive fusion tied the weaker of the two")
    print("   proper methods — still never normalize raw scores in production, this")
    print("   dataset is just too small to always expose the gap.")
else:
    print(
        "\n-> Surprising on this tiny validation set: naive fusion didn't underperform"
    )
    print(
        "   here, but that's a property of this specific 5-query sample, not a reason"
    )
    print("   to skip normalization — BM25's unbounded scale is still a latent bug.")


---

## Part 6 — Score Normalization: Making Scores Comparable

**Riverside's question for this section:** if we do want to blend raw scores instead of ranks, how
do we make cosine similarity and BM25 scores comparable in the first place?

When using weighted score fusion, scores from different systems must be scaled to the same range before blending. Two common techniques:

**Min-Max Normalization** maps every score to [0, 1] by measuring its position between the observed min and max: $\text{norm}(x) = \frac{x - \min(X)}{\max(X) - \min(X)}$

The intuition: the best-performing document in the result set scores 1.0, the worst scores 0.0, and everything else falls proportionally in between. The weakness is sensitivity to outliers — one extremely high BM25 score stretches the denominator and compresses all other scores toward 0.

**Z-Score Normalization** expresses each score as a distance from the mean in standard deviation units: $z(x) = \frac{x - \mu}{\sigma}$

The intuition: outliers produce large z-scores but don't compress the rest of the distribution. The tradeoff is that z-scores can be negative, which requires extra handling (e.g., shifting by the minimum) before using them as blend weights.

For hybrid search, min-max is more common because its [0, 1] output maps cleanly onto α-weighted blending. RRF sidesteps this entire problem by discarding scores altogether and working only with ranks.

Let's visualize how normalization affects score distributions:


In [ ]:
# Score-normalization visualization.


def z_score_normalize(scores):
    """Z-score normalization (mean=0, standard deviation=1)."""
    scores = np.array(scores)
    mean = scores.mean()
    std = scores.std()
    if std == 0:
        return np.zeros_like(scores)
    return (scores - mean) / std


query, [normalization_target] = POLICY_QUERY_FIXTURE["expense_deadline"]
sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)

sem_scores_raw = [score for _, score in sem_results]
lex_scores_raw = [score for _, score in lex_results]

sem_scores_minmax = min_max_normalize(sem_scores_raw)
sem_scores_zscore = z_score_normalize(sem_scores_raw)
lex_scores_minmax = min_max_normalize(lex_scores_raw)
lex_scores_zscore = z_score_normalize(lex_scores_raw)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f"Riverside Score Normalization\nQuery: {query!r}; target policy: {normalization_target}",
    fontsize=14,
    fontweight="bold",
)

axes[0, 0].bar(range(10), sem_scores_raw, color="steelblue", alpha=0.7)
axes[0, 0].set_title("Semantic Scores (Raw)")
axes[0, 0].set_ylabel("Cosine Similarity")

axes[0, 1].bar(range(10), sem_scores_minmax, color="green", alpha=0.7)
axes[0, 1].set_title("Semantic Scores (Min-Max)")
axes[0, 1].set_ylim([0, 1])

axes[0, 2].bar(range(10), sem_scores_zscore, color="purple", alpha=0.7)
axes[0, 2].set_title("Semantic Scores (Z-Score)")
axes[0, 2].axhline(0, color="red", linestyle="--", linewidth=1)

axes[1, 0].bar(range(10), lex_scores_raw, color="coral", alpha=0.7)
axes[1, 0].set_title("Lexical Scores (Raw BM25)")
axes[1, 0].set_ylabel("BM25 Score")
axes[1, 0].set_xlabel("Policy Rank Position")

axes[1, 1].bar(range(10), lex_scores_minmax, color="green", alpha=0.7)
axes[1, 1].set_title("Lexical Scores (Min-Max)")
axes[1, 1].set_xlabel("Policy Rank Position")
axes[1, 1].set_ylim([0, 1])

axes[1, 2].bar(range(10), lex_scores_zscore, color="purple", alpha=0.7)
axes[1, 2].set_title("Lexical Scores (Z-Score)")
axes[1, 2].set_xlabel("Policy Rank Position")
axes[1, 2].axhline(0, color="red", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

print("\nNormalization observations:")
print("- Min-max maps each result list to [0, 1] but is sensitive to extreme values.")
print("- Z-scores center each list at zero but can be negative.")
print("- RRF avoids both transformations by using ranks instead of raw scores.")


---

## Part 7 — Alpha Tuning: Finding a Defensible Balance

**Riverside's question for this section:** is there one semantic/lexical blend ratio that works across exact policy codes, formal controls, and everyday employee questions?

The parameter **α** controls the blend: $\text{hybrid\_score} = (1 - \alpha) \cdot \text{lexical} + \alpha \cdot \text{semantic}$.

The interpretation is direct: α=0 is pure BM25, α=1 is pure dense retrieval, and intermediate values weight both. A query mix dominated by exact identifiers may favor a lower α; a mix dominated by paraphrases may favor a higher one. Riverside's mixture includes both, so intuition alone cannot select the value.

**How to choose α:**

1. **Query mix:** include codes, named controls, and employee paraphrases.
2. **Validation set:** test candidate α values on fixed query-policy labels.
3. **Held-out check:** evaluate the selected α on different questions before release.
4. **Production review:** revalidate as policies and employee language change.

### Validation Experiment: Alpha Sweep

Use the five canonical Riverside validation queries from the shared fixture, sweep α from 0 to 1, and measure Recall@5. On ten documents, Recall@5 may be too forgiving to separate the settings; the code reports that honestly if the curve is flat.


#### So we can normalize scores — but what is the right α?

Normalization makes cosine and BM25 values comparable, but it does not decide how much Riverside should trust each signal.

- α = 0 → pure lexical retrieval.
- α = 0.5 → equal semantic and lexical weights.
- α = 1 → pure semantic retrieval.

The validation fixture deliberately mixes exact and paraphrased policy questions. The sweep below measures that specific mix; it does not establish a permanent company-wide constant. A flat Recall@5 curve means the toy corpus and cutoff cannot distinguish the candidates, not that the first α returned by `argmax` is best.


In [ ]:
# Alpha-tuning experiment over the shared Riverside validation fixture.
alpha_values = np.linspace(0, 1, 21)
recall_scores = []

for alpha in alpha_values:
    recalls = []
    for query, relevant_docs in validation_queries:
        sem_results = semantic_search(query, top_k=10)
        lex_results = lexical_search(query, top_k=10)
        hybrid_results = weighted_score_fusion(sem_results, lex_results, alpha=alpha)
        recalls.append(compute_recall_at_k(hybrid_results, relevant_docs, k=5))
    recall_scores.append(np.mean(recalls))

optimal_alpha = alpha_values[np.argmax(recall_scores)]
max_recall = max(recall_scores)
recall_spread = max(recall_scores) - min(recall_scores)

plt.figure(figsize=(12, 6))
plt.plot(
    alpha_values,
    recall_scores,
    "o-",
    linewidth=2,
    markersize=6,
    color="steelblue",
)
plt.axvline(
    optimal_alpha,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"First max α={optimal_alpha:.2f} (Recall@5={max_recall:.2f})",
)
plt.xlabel("Alpha (Semantic Weight)")
plt.ylabel("Average Recall@5")
plt.title("Riverside Policy Alpha Sweep\nFive canonical validation queries")
plt.xticks(alpha_values[::2], [f"{value:.1f}" for value in alpha_values[::2]])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nFirst α at maximum Recall@5: {optimal_alpha:.2f}")
print(f"Maximum Recall@5: {max_recall:.2f}")
print("\nBaseline comparisons:")
print(f"  α=0.0 (pure lexical):  Recall@5={recall_scores[0]:.2f}")
print(f"  α=0.5 (balanced):      Recall@5={recall_scores[10]:.2f}")
print(f"  α=1.0 (pure semantic): Recall@5={recall_scores[-1]:.2f}")
print("\nInterpretation:")

if recall_spread < 0.05:
    print(
        f"  [INCONCLUSIVE] Recall@5 is effectively flat "
        f"({min(recall_scores):.2f}-{max(recall_scores):.2f}) across the sweep."
    )
    print("  The reported α is only the first maximum selected by argmax, not evidence of superiority.")
    print("  Use a larger corpus, stricter cutoff, and more discriminating labels before choosing α.")
elif optimal_alpha < 0.3:
    print(f"  This validation mix favored lexical weight (first maximum α={optimal_alpha:.2f}).")
elif optimal_alpha > 0.7:
    print(f"  This validation mix favored semantic weight (first maximum α={optimal_alpha:.2f}).")
else:
    print(f"  This validation mix favored a blended setting (first maximum α={optimal_alpha:.2f}).")
print("\nFor production, tune on a larger labeled policy-query set and confirm on held-out queries.")


### Your turn — manually tune α

The sweep above identified the first maximum on the validation set; it may not have identified a unique optimum. Change `alpha_manual` below and inspect how the ranking changes for `support when welcoming a baby`, whose labeled target is policy 5 (Parental Leave).

**Predict before running:** will pure lexical retrieval find enough overlap to rank policy 5 first? Will increasing the semantic weight improve, preserve, or worsen the target rank on this actual run?


In [ ]:
# Your turn — blend ratio alpha.
# Change alpha_manual (0.0 = pure lexical, 1.0 = pure semantic).

alpha_manual = 0.5

query_tune, [query_tune_target] = POLICY_QUERY_FIXTURE["parental_leave"]
sem_tune = semantic_search(query_tune, top_k=10)
lex_tune = lexical_search(query_tune, top_k=10)

sem_norm = min_max_normalize([score for _, score in sem_tune])
lex_norm = min_max_normalize([score for _, score in lex_tune])

sem_scored = [
    (index, alpha_manual * score) for (index, _), score in zip(sem_tune, sem_norm)
]
lex_scored = [
    (index, (1 - alpha_manual) * score)
    for (index, _), score in zip(lex_tune, lex_norm)
]

all_scores = {}
for index, score in sem_scored + lex_scored:
    all_scores[index] = all_scores.get(index, 0) + score
all_blended = sorted(all_scores.items(), key=lambda item: item[1], reverse=True)
blended = all_blended[:5]

print(
    f"α={alpha_manual:.1f} blend (query: {query_tune!r}; "
    f"target policy: {query_tune_target})"
)
print(
    f"  {int((1-alpha_manual)*100)}% lexical weight + "
    f"{int(alpha_manual*100)}% semantic weight"
)
for rank, (doc_idx, score) in enumerate(blended, 1):
    marker = "  <-- target" if doc_idx == query_tune_target else ""
    print(f"  Rank {rank}: Policy {doc_idx} score={score:.3f}{marker}")
    print(f"    {documents[doc_idx][:90]}...")

target_rank = next(
    rank for rank, (index, _) in enumerate(all_blended, 1) if index == query_tune_target
)
semantic_target_rank = next(
    rank for rank, (index, _) in enumerate(sem_tune, 1) if index == query_tune_target
)
lexical_target_rank = next(
    rank for rank, (index, _) in enumerate(lex_tune, 1) if index == query_tune_target
)
print(
    f"\nMeasured target ranks — lexical: {lexical_target_rank}; "
    f"semantic: {semantic_target_rank}; blended: {target_rank}."
)


### Common Pitfalls: Alpha Tuning

**Pitfall #1: Hardcoding α=0.5 “to be safe”**

**Bad:** choose a round-number α without validating it against Riverside's labeled questions.
**Good:** sweep α over a query mix that includes policy codes, formal controls, and employee paraphrases. If the curve is flat, report that the data cannot choose.

**Pitfall #2: Treating one small validation set as permanent**

**Bad:** tune on five launch questions and never revisit the value as policies, terminology, and employee behavior change.
**Good:** keep a held-out set, add judged production questions over time, and treat the selected α as a hypothesis to recheck.

**Quick health check:** evaluate the selected α on the three distinct held-out fixture queries about records retention, workplace reporting, and travel receipts. Report a win, tie, or loss against α=0.5 exactly as measured.


In [ ]:
# Held-out health check: does the selected alpha generalize to new policy questions?


def recall_at_alpha(alpha, queries):
    """Average Recall@5 for an alpha over query-policy labels."""
    recalls = []
    for query, relevant_docs in queries:
        sem_results = semantic_search(query, top_k=10)
        lex_results = lexical_search(query, top_k=10)
        hybrid_results = weighted_score_fusion(sem_results, lex_results, alpha=alpha)
        recalls.append(compute_recall_at_k(hybrid_results, relevant_docs, k=5))
    return np.mean(recalls)


held_out_at_optimal = recall_at_alpha(optimal_alpha, held_out_queries)
held_out_at_default = recall_at_alpha(0.5, held_out_queries)

print("Held-out Riverside policy queries:")
for query, targets in held_out_queries:
    print(f"- {query!r} -> target policy {targets[0]}")

print(f"\nRecall@5 at selected α={optimal_alpha:.2f}: {held_out_at_optimal:.3f}")
print(f"Recall@5 at default α=0.50:       {held_out_at_default:.3f}")

if held_out_at_optimal > held_out_at_default:
    print("\nThe selected α scored higher on these three held-out questions.")
elif held_out_at_optimal < held_out_at_default:
    print("\nThe selected α scored lower than the default on these three held-out questions.")
else:
    print("\nThe selected α tied the default on these three held-out questions.")
print("Three held-out labels remain too small to lock a production setting.")


---

## Part 8 — LangChain Components: Production-Style Ensemble Wiring

**Riverside's question for this section:** how can the IT team expose the same policy corpus through a vector index, a BM25 retriever, and one fused interface?

The next cell uses LangChain document, embedding, FAISS, and BM25 components, then wraps them in a small `EnsembleRetriever` class whose behavior is visible in the notebook. Its weighted RRF path:

1. invokes each retriever independently;
2. converts each ranked position to `weight × 1/(60 + rank)`;
3. sums contributions for policy passages returned by both retrievers;
4. returns the top fused documents.

This is rank fusion, not min-max score averaging. The explicit implementation keeps the production contract inspectable while avoiding a dependency on a particular LangChain ensemble module version.


> **PyTorch → Keras:** `HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` is a LangChain wrapper that internally loads the same PyTorch `sentence-transformers` model used earlier and exposes it through LangChain's `Embeddings` interface; `FAISS.from_documents(...)` then calls that wrapper to embed every document and builds a FAISS similarity index over the resulting PyTorch-produced vectors. **Keras/TF equivalent:** LangChain has no first-class Keras/TF embeddings wrapper — the nearest equivalent is a small custom `Embeddings` subclass that calls a `tf.keras`/TF-Hub encoder inside `embed_documents`/`embed_query`, then passing that custom wrapper into `FAISS.from_documents` the same way.


In [ ]:
# Production-style hybrid search pipeline using LangChain components.
from langchain_core.documents import Document
from langchain_community.retrievers.bm25 import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


class EnsembleRetriever:
    """Minimal weighted-RRF ensemble retriever."""

    def __init__(self, retrievers, weights=None):
        self.retrievers = retrievers
        self.weights = weights or [1.0 / len(retrievers)] * len(retrievers)

    def invoke(self, query: str, k: int = 5):
        rrf = {}
        for retriever, weight in zip(self.retrievers, self.weights):
            docs = retriever.invoke(query)
            for rank, doc in enumerate(docs, 1):
                key = doc.page_content
                rrf[key] = rrf.get(key, {"score": 0.0, "doc": doc})
                rrf[key]["score"] += weight * (1.0 / (60 + rank))
        ranked = sorted(rrf.values(), key=lambda item: item["score"], reverse=True)
        return [item["doc"] for item in ranked[:k]]


langchain_docs = [
    Document(page_content=doc, metadata={"doc_id": index})
    for index, doc in enumerate(documents)
]

print("Initializing embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("[OK]\n")

print("Building FAISS vector store...")
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("[OK]\n")

print("Building BM25 retriever...")
bm25_retriever = BM25Retriever.from_documents(langchain_docs)
bm25_retriever.k = 5
print("[OK]\n")

print("Creating weighted-RRF ensemble (weights=[0.3, 0.7])...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.3, 0.7],
)
print("[OK]\n")

ensemble_query_keys = ["rights_code", "parental_leave", "public_ai"]
test_queries_ensemble = [POLICY_QUERY_FIXTURE[key] for key in ensemble_query_keys]

for query, target_policies in test_queries_ensemble:
    print(f"Query: {query!r}; labeled target: policy {target_policies[0]}")
    print("-" * 80)
    results = ensemble_retriever.invoke(query)

    for rank, doc in enumerate(results[:3], 1):
        policy_id = doc.metadata["doc_id"]
        marker = "  <-- target" if policy_id in target_policies else ""
        print(f"Rank {rank}: Policy {policy_id}{marker}")
        print(f"  {doc.page_content[:110]}...\n")

    target_in_top_three = any(
        doc.metadata["doc_id"] in target_policies for doc in results[:3]
    )
    print(f"Target present in top 3: {target_in_top_three}\n")

print("Policy-query fixture used by this ensemble:")
print(f"{'Query':<48} {'Target policy':<14} {'Policy area'}")
print("-" * 85)
policy_areas = {
    0: "Copyright / RIGHTS-17",
    2: "AI Usage",
    3: "Travel and Expenses",
    5: "Parental Leave",
    6: "Procurement / FIN-42",
}
for key in [
    "rights_code",
    "parental_leave",
    "public_ai",
    "expense_deadline",
    "supplier_quotes",
]:
    query, [target] = POLICY_QUERY_FIXTURE[key]
    print(f"{query:<48} {target:<14} {policy_areas[target]}")


### Code Walkthrough: Ensemble Retrieval

The production-shaped example reuses the same Riverside corpus behind standard interfaces:

- `HuggingFaceEmbeddings` exposes the dense encoder to FAISS.
- `BM25Retriever` exposes lexical search through the same retriever API.
- `EnsembleRetriever.invoke()` runs both and adds weighted reciprocal-rank contributions.
- Documents retain their `doc_id`, which is the minimum metadata needed to trace a result back to its source.

Because the corpus itself is Riverside policy text, the final query table is no longer an analogy. It shows the exact employee questions this local retriever is intended to serve.

> **API note:** `EnsembleRetriever.invoke` returns `Document` objects rather than `(index, score)` tuples. Read `.page_content` for text and `.metadata["doc_id"]` for the source identifier.

---

## Part 9 - Optional Production Patterns

The ten-policy corpus is tiny. As Riverside indexes policy histories, contracts, employee handbooks, and manuscript permissions, two pressures appear: searching every vector becomes slower, and independently encoded query/document vectors may miss fine distinctions.

### Two-stage retrieval

Use a fast first stage to collect candidates, then spend a more expensive model only on that shortlist:

1. retrieve a broad candidate set with BM25 or hybrid search;
2. rerank those candidates with a dense model or cross-encoder;
3. return only the best few passages.

This notebook demonstrates the pipeline shape. It does not claim a speedup from ten documents; production latency must be measured on Riverside's real index.

### Query rewriting

An employee may ask `Can I use a draft in a public chatbot?` while the approved text says `public generative AI services`. Rewriting or synonym expansion can reduce that vocabulary gap, but it also adds another model and another place for meaning to drift. Use labeled Riverside questions before enabling it.

### Domain-specific embeddings

General embeddings may miss legal, HR, or publishing distinctions. Riverside could compare a legal-language encoder for copyright and contracts or a finance-oriented encoder for procurement. A model name alone proves nothing; the replacement must win on held-out Riverside relevance judgments.

### Exact versus approximate vector indexing

Current searches compare a query with every document vector. At much larger scale, IVF searches only promising clusters and HNSW walks a proximity graph. Both trade some recall for speed and therefore require the same labeled retrieval tests used elsewhere in this notebook.

The next cells demonstrate two-stage retrieval and cross-encoder reranking on Riverside policy questions.

> **PyTorch → Keras:** the same PyTorch sentence-transformer is called twice here — once to embed only the BM25-narrowed `candidate_docs` (not the full corpus) and once to embed the query — reusing the model already loaded earlier rather than reloading it. **Keras/TF equivalent:** the identical pattern with a `tf.keras`/TF-Hub encoder, calling it on the smaller candidate subset instead of the full document set to save inference cost during the second stage.


In [ ]:
# Two-stage retrieval: BM25 candidate collection followed by dense reranking.
def two_stage_retrieval(query, stage1_k=10, stage2_k=5):
    """Retrieve BM25 candidates, then rerank only those candidates semantically."""
    stage1_results = lexical_search(query, top_k=min(stage1_k, len(documents)))
    candidate_indices = [index for index, _ in stage1_results]

    candidate_documents = [documents[index] for index in candidate_indices]
    candidate_embeddings = semantic_model.encode(
        candidate_documents, show_progress_bar=False
    )
    query_embedding = semantic_model.encode([query], show_progress_bar=False)
    similarities = cosine_similarity(query_embedding, candidate_embeddings)[0]
    reranked_positions = np.argsort(similarities)[::-1][:stage2_k]
    return [
        (candidate_indices[position], similarities[position])
        for position in reranked_positions
    ]


query = "purchase approval supplier quotes"
two_stage_results = two_stage_retrieval(query)

print(f"Query: {query!r}\n")
for rank, (index, score) in enumerate(two_stage_results, 1):
    marker = "  <-- procurement policy" if index == 6 else ""
    print(f"Rank {rank} (dense rerank={score:.3f}): Document {index + 1}{marker}")
    print(f"  {documents[index]}\n")

print("This toy run demonstrates candidate -> rerank control flow, not a measured speedup.")

### Cross-Encoder Reranking

Dense retrieval encodes the query and document separately, which is efficient but limits token-level interaction. A cross-encoder reads each `(query, document)` pair together and can make finer relevance distinctions.

That extra attention is expensive, so use it only after hybrid retrieval has narrowed the corpus. The next cell reranks five Riverside policy candidates and reports honestly whether their order changes; a changed order is not automatically an improvement unless the relevant policy moves upward.

> **PyTorch → Keras:** `CrossEncoder(...)` loads a different PyTorch transformer architecture than the bi-encoder above — it takes a `(query, document)` pair as one joint input sequence — and `.predict(pairs)` runs a batched forward pass returning one relevance score per pair. **Keras/TF equivalent:** the closest TF pattern is a `TFAutoModelForSequenceClassification` loaded from the same or an equivalent cross-encoder checkpoint, tokenizing `(query, document)` pairs together and calling the Keras model directly (or `.predict()` on a `tf.data.Dataset` of pairs) to get the same per-pair scores.


In [ ]:
# Cross-encoder reranking on a Riverside employee question.
from sentence_transformers import CrossEncoder

print("Loading cross-encoder model (ms-marco-MiniLM-L6-v2)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

query_rerank = "can staff paste draft novels into ChatGPT"
relevant_policy = 2
semantic_candidates = semantic_search(query_rerank, top_k=10)
lexical_candidates = lexical_search(query_rerank, top_k=10)
hybrid_candidates = reciprocal_rank_fusion(
    semantic_candidates, lexical_candidates, k=60
)[:5]

pairs = [(query_rerank, documents[index]) for index, _ in hybrid_candidates]
cross_scores = cross_encoder.predict(pairs)
reranked = sorted(
    zip([index for index, _ in hybrid_candidates], cross_scores),
    key=lambda item: float(item[1]),
    reverse=True,
)

print(f"Query: {query_rerank!r}\n")
print("Before reranking (hybrid RRF):")
for rank, (index, score) in enumerate(hybrid_candidates, 1):
    marker = "  <-- AI usage policy" if index == relevant_policy else ""
    print(f"  {rank}. Document {index + 1} (RRF={score:.4f}){marker}")

print("\nAfter cross-encoder reranking:")
for rank, (index, score) in enumerate(reranked, 1):
    old_rank = next(
        position + 1
        for position, (candidate_index, _) in enumerate(hybrid_candidates)
        if candidate_index == index
    )
    marker = "  <-- AI usage policy" if index == relevant_policy else ""
    print(
        f"  {rank}. Document {index + 1} (cross-encoder={float(score):.3f}, "
        f"was {old_rank}){marker}"
    )

old_relevant_rank = next(
    (rank for rank, (index, _) in enumerate(hybrid_candidates, 1) if index == relevant_policy),
    None,
)
new_relevant_rank = next(
    (rank for rank, (index, _) in enumerate(reranked, 1) if index == relevant_policy),
    None,
)
print(f"\nRelevant policy rank: {old_relevant_rank} -> {new_relevant_rank}")
print("Judge reranking by relevant-policy movement across many labeled questions, not one changed order.")

---

## Part 10 — Benchmarking: Measuring Retrieval Quality

**Riverside's question for this section:** before this ships to every employee, how do we know it's
actually better than what people do today (asking a colleague, or re-reading chapters)?

How do we know if hybrid search is actually better? We need **quantitative metrics** on a labeled test set.

### Key Metrics

**1. Recall@K**

Recall@K measures what fraction of all relevant documents appear in the top-K results: $\text{Recall@K} = \frac{\text{relevant docs in top-K}}{\text{total relevant docs}}$

The intuition: a system that retrieves all 3 relevant documents in its top-5 results scores 1.0; one that retrieves only 1 of 3 scores 0.33. Higher K gives the retriever more chances — Recall@10 is always at least as high as Recall@5.

**2. Mean Reciprocal Rank (MRR)**

MRR rewards systems that surface the first relevant document as high as possible: $\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$

The reciprocal is the key: a relevant document at rank 1 contributes 1.0 to the average, at rank 2 contributes 0.5, at rank 10 only 0.1. A system that always finds the right document but buries it at rank 5 scores much lower than one that consistently puts it first.

**3. Normalized Discounted Cumulative Gain (NDCG@K)**

For cases where relevance is graded (not binary), NDCG@K rewards putting the most relevant items first: $\text{NDCG@K} = \text{DCG@K} / \text{IDCG@K}$

where DCG sums logarithmically-discounted relevance scores and IDCG is the DCG of the ideal ordering. For binary relevance (as in this notebook), Recall@K and MRR are sufficient.

### Benchmark Experiment

Let's compute Recall@5 and MRR for semantic, lexical, and hybrid search on our validation set:


### Bridging from Toy to Production

The algorithms and fusion techniques we've implemented are **identical** in production — only the infrastructure changes. Here's how our toy example maps to real-world scale:

| Parameter              | Toy (this notebook)       | Production                            | Notes                                                  |
| ---------------------- | ------------------------- | ------------------------------------- | ------------------------------------------------------ |
| **Corpus size**        | 10 docs                   | 100K – 10M docs                       | BM25 stays fast; vector search needs HNSW/IVF indexing |
| **Embedding model**    | `all-MiniLM-L6-v2` (384D) | `all-MiniLM-L6-v2` or domain-specific | Same code, just swap the model                         |
| **BM25 top-k**         | 10 (all docs)             | 100 – 1000                            | Stage 1 pre-filter in two-stage retrieval              |
| **Semantic top-k**     | 5 – 10                    | 5 – 20                                | Final results returned to user                         |
| **RRF constant k**     | 60                        | 60                                    | Empirically robust across domains                      |
| **α (sem/lex weight)** | Tuned on 5 queries        | Tuned on 50-100 labeled pairs         | Validation set size scales with domain diversity       |
| **Recall@K metric**    | K=5                       | K=5, 10, 20                           | Larger K for exploratory retrieval                     |
| **Latency**            | ~10ms (toy)               | ~50-200ms (two-stage, reranking)      | Add caching for repeated queries                       |

**Key Insight**: The **algorithms and fusion techniques are identical**. Only the infrastructure changes:

- **Vector indexing**: FAISS, Pinecone, Weaviate for fast approximate nearest neighbor search
- **Caching**: Redis/Memcached for repeated queries
- **Batch inference**: Group queries to amortize embedding overhead
- **Reranking**: Cross-encoder on top-20 candidates for final quality boost


In [ ]:
#  Benchmark Experiment: Recall@5 and MRR


def compute_mrr(results, relevant_docs):
    """Compute Mean Reciprocal Rank"""
    for rank, (idx, _) in enumerate(results, start=1):
        if idx in relevant_docs:
            return 1.0 / rank
    return 0.0


# Wrap each retrieval strategy behind the same (query) -> [(idx, score)] signature
methods = [
    ("Semantic", lambda q: semantic_search(q, top_k=10)),
    ("Lexical (BM25)", lambda q: lexical_search(q, top_k=10)),
    (
        "Hybrid (RRF)",
        lambda q: reciprocal_rank_fusion(
            semantic_search(q, top_k=10), lexical_search(q, top_k=10), k=60
        ),
    ),
]

results_table = []

# Measure Recall@5 and MRR for every method across the whole validation set
for method_name, search_fn in methods:
    recall_scores = []
    mrr_scores = []

    for query, relevant_docs in validation_queries:
        results = search_fn(query)
        recall = compute_recall_at_k(results, relevant_docs, k=5)
        mrr = compute_mrr(results, relevant_docs)

        recall_scores.append(recall)
        mrr_scores.append(mrr)

    avg_recall = np.mean(recall_scores)
    avg_mrr = np.mean(mrr_scores)

    results_table.append(
        {
            "Method": method_name,
            "Recall@5": f"{avg_recall:.3f}",
            "MRR": f"{avg_mrr:.3f}",
        }
    )

df_benchmark = pd.DataFrame(results_table)

print("Benchmark Results on Validation Set")
print("=" * 80)
print(df_benchmark.to_string(index=False))
print("\nKey Takeaways:")
print("   - Hybrid search typically outperforms either method alone")
print("   - Recall@5 measures coverage (did we find relevant docs?)")
print("   - MRR measures ranking quality (did we rank relevant docs high?)")
print("   - These metrics prove hybrid combines the strengths of both approaches")


### Code Walkthrough: Benchmark — Recall@5 and MRR

**What just ran — 3 key patterns:**

---

**`compute_mrr(results, relevant_docs)` — reciprocal of the first relevant rank**
The function iterates through the ranked result list and returns `1/rank` the moment it finds a relevant document. A correct document at rank 1 contributes 1.0 to the average; at rank 2, 0.5; at rank 10, only 0.1. This steep decay means MRR strongly rewards consistently top-ranked correct results — exactly the requirement for Riverside employees who type a character name and expect that document first, not buried at rank 5.

---

**Lambda wrappers in `methods` — polymorphic search interface**
Each entry stores a `(name, lambda)` pair wrapping a different search function with a uniform `search_fn(query) → [(idx, score)]` signature. The benchmark loop iterates identically over semantic, lexical, and hybrid without branching. Adding a new retriever variant (cross-encoder reranking, query expansion) means adding one list entry, not rewriting the loop.

---

**`df_benchmark` print block — honest multi-method comparison**
The output says hybrid "typically" outperforms rather than "always" — because on this 10-document corpus the recall plateau (top-5 out of 10) limits the measurable advantage. The honest framing matters: a student reading the output should understand both what hybrid search offers and why a toy corpus cannot prove it conclusively. The dual heatmap further down provides the per-query view of where each method wins and loses.

> **Shape/API note:** `compute_recall_at_k` and `compute_mrr` both expect `results` as `[(idx, score), ...]`. Pass the same list to both; neither modifies the input.


### Compare Dense, BM25, and Hybrid by Question

All three strategies use the same five Riverside validation questions. The aggregate table can hide where a method succeeds, so the heatmaps show each question separately.

| Strategy | Usually strongest when | Main blind spot |
| --- | --- | --- |
| Dense | Employee wording paraphrases formal policy language | Exact identifiers such as `RIGHTS-17` |
| BM25 | Codes, names, and approved phrases must match exactly | Meaning expressed with different words |
| Hybrid RRF | Riverside receives both query types | Extra retrieval work; fusion still needs validation |

Recall@5 shows whether all labeled passages were found. MRR shows how early the first relevant policy appeared.

In [ ]:
#  Multi-Strategy Comparison Dual Heatmap
# Pattern 3 — dual heatmap: retrieval method × query, two metrics (Recall@5 and MRR)

methods_grid = ["Semantic", "Lexical (BM25)", "Hybrid (RRF)"]
n_m, n_q_g = len(methods_grid), len(validation_queries)

# Pre-fill with NaN so any un-computed cell renders as grey ("not available")
recall_grid = np.full((n_m, n_q_g), np.nan)
mrr_grid = np.full((n_m, n_q_g), np.nan)

# Same three retrieval strategies as the benchmark cell, wrapped for grid population
_search_fns = [
    lambda q: semantic_search(q, top_k=10),
    lambda q: lexical_search(q, top_k=10),
    lambda q: reciprocal_rank_fusion(
        semantic_search(q, top_k=10), lexical_search(q, top_k=10), k=60
    ),
]

# Fill the method x query grid with Recall@5 and MRR for every combination
for mi, fn in enumerate(_search_fns):
    for qi, (qry, rel) in enumerate(validation_queries):
        res = fn(qry)
        recall_grid[mi, qi] = compute_recall_at_k(res, rel, k=5)
        mrr_grid[mi, qi] = compute_mrr(res, rel)

# Grey out any NaN cell instead of letting matplotlib guess a color for it
cmap_hm = plt.cm.YlGn.copy()
cmap_hm.set_bad(color="#d3d3d3")

fig_hm, (ax_r, ax_m) = plt.subplots(1, 2, figsize=(13, 3.5))

# Draw one heatmap per metric (Recall@5, MRR), sharing the same grey-for-NaN style
for ax, mat, title in [(ax_r, recall_grid, "Recall@5"), (ax_m, mrr_grid, "MRR")]:
    im = ax.imshow(mat, cmap=cmap_hm, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(n_q_g))
    ax.set_xticklabels([f"Q{i+1}" for i in range(n_q_g)])
    ax.set_yticks(range(n_m))
    ax.set_yticklabels(methods_grid)
    ax.set_title(f"{title} — method × query")

    # Annotate each cell with its numeric value (or a dash if not available)
    for i in range(n_m):
        for j in range(n_q_g):
            v = mat[i, j]
            ax.text(
                j,
                i,
                f"{v:.2f}" if not np.isnan(v) else "—",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if v > 0.65 else "black",
            )
    fig_hm.colorbar(im, ax=ax)

plt.suptitle(
    "Dual Heatmap: Retrieval Strategy × Query\n(grey = not available)",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print(
    f"Mean Recall@5 — Semantic: {np.nanmean(recall_grid[0]):.3f}  "
    f"Lexical: {np.nanmean(recall_grid[1]):.3f}  "
    f"Hybrid: {np.nanmean(recall_grid[2]):.3f}"
)
print(
    f"Mean MRR      — Semantic: {np.nanmean(mrr_grid[0]):.3f}  "
    f"Lexical: {np.nanmean(mrr_grid[1]):.3f}  "
    f"Hybrid: {np.nanmean(mrr_grid[2]):.3f}"
)
print("\nHybrid RRF covers each method's blind spot: where Semantic misses (rare")
print("terms), BM25 contributes, and where BM25 misses (synonyms), semantic steps in.")


### Visualizing Performance Across Queries

Let's see which queries benefit most from hybrid search:


In [ ]:
#  Per-query recall breakdown — animated (FuncAnimation)
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from IPython.display import HTML, display

query_names = [q for q, _ in validation_queries]
semantic_recalls = []
lexical_recalls = []
hybrid_recalls = []

# Compute Recall@5 for all three methods on every validation query, for the animation
for query, relevant_docs in validation_queries:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    hyb_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
    semantic_recalls.append(compute_recall_at_k(sem_results, relevant_docs, k=5))
    lexical_recalls.append(compute_recall_at_k(lex_results, relevant_docs, k=5))
    hybrid_recalls.append(compute_recall_at_k(hyb_results, relevant_docs, k=5))

#  Animated grouped bar chart — one query per frame
n_q = len(query_names)
width = 0.25

fig_anim, ax_anim = plt.subplots(figsize=(13, 5))


# Redraw every query bar group up to the current frame, fading all but the newest
def _draw_recall_frame(frame):
    ax_anim.clear()
    for qi in range(frame + 1):
        alpha = 1.0 if qi == frame else 0.45
        ax_anim.bar(
            qi - width, semantic_recalls[qi], width, color="steelblue", alpha=alpha
        )
        ax_anim.bar(qi, lexical_recalls[qi], width, color="coral", alpha=alpha)
        ax_anim.bar(qi + width, hybrid_recalls[qi], width, color="green", alpha=alpha)
    ax_anim.set_xlim(-0.7, n_q - 0.3)
    ax_anim.set_ylim(0, 1.25)
    ax_anim.set_xticks(range(n_q))
    ax_anim.set_xticklabels([f"Q{i+1}" for i in range(n_q)])
    ax_anim.set_ylabel("Recall@5")
    ax_anim.set_xlabel("Query")
    ax_anim.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.4)
    ax_anim.set_title(f"Recall@5 per Query — Q{frame+1}: '{query_names[frame][:35]}'")
    ax_anim.grid(axis="y", alpha=0.3)

    # Static legend, redrawn every frame (ax_anim.clear() above wipes it otherwise) —
    # it must be visible from frame 0, not only appear once the animation finishes.
    ax_anim.legend(
        handles=[
            Patch(facecolor="steelblue", label="Semantic"),
            Patch(facecolor="coral", label="Lexical (BM25)"),
            Patch(facecolor="green", label="Hybrid (RRF)"),
        ],
        loc="upper right",
    )


anim = FuncAnimation(
    fig_anim, _draw_recall_frame, frames=n_q, interval=800, repeat=False
)
plt.close(fig_anim)

print("Animated Recall@5 comparison — one query revealed per frame.")
print("Watch which method tops each bar: Hybrid (green) fills the gaps that")
print("Semantic (blue) and Lexical (coral) each leave on different queries.\n")
display(HTML(anim.to_jshtml(fps=6)))

print("\nQuery Details:")
for i, query in enumerate(query_names, 1):
    print(f"Q{i}: {query}")


---

## Summary: Answer the Opening Question

**Can Riverside answer corpus-specific questions without fine-tuning facts into the model?**

Yes, if the system retrieves current evidence at request time and the generator is required to answer from that evidence. Fine-tuning remains useful for persistent behavior such as style and instruction following; it is not the corpus database.

### What the Failure Chain Established

| Failure | Minimal response | Remaining issue |
| --- | --- | --- |
| Model weights cannot provide current, citable facts reliably | Retrieve passages at request time | Dense retrieval can bury rare exact terms |
| Dense search misses an identifier | Add BM25 lexical retrieval | BM25 misses paraphrases with no token overlap |
| Dense and lexical return complementary ranked lists | Fuse them with RRF | Retrieval quality still needs labeled evaluation |
| One fusion choice looks plausible | Measure Recall@$K$ and MRR on fixed queries | Small toy labels do not establish production performance |

### Practical Intuition

- Dense retrieval answers “what passage means something like this?”
- BM25 answers “what passage contains this exact term?”
- RRF combines ranks when the score scales are incompatible.
- Normalized weighted fusion is useful only when Riverside has enough labels to tune and revalidate $\alpha$.
- Cross-encoder reranking is a second stage for a small candidate set, not a replacement for first-stage retrieval.

### What Was Measured

The notebook computes semantic, lexical, RRF, weighted-fusion, and naive-fusion results on the same labeled queries, then reports Recall@5 and MRR. Read the scorecard that follows rather than memorizing an asserted winner. The exact values belong to the ten-document teaching corpus.

### What Riverside Still Must Prove

1. Re-index the actual manuscript corpus with stable chunk IDs and access-control metadata.
2. Label representative employee questions, including exact names, paraphrases, and no-answer cases.
3. Re-measure retrieval quality and latency on that corpus.
4. Pass retrieved passages into the accepted assistant with citations.
5. Evaluate grounding and answer correctness in the next RAG evaluation notebook.

> **Handoff:** this chapter proves whether the supporting passage was retrieved. The next chapter asks whether the generated answer actually uses that passage correctly.

In [ ]:
#  Scorecard: pulling together every metric measured in this notebook
# Every value below comes from a variable already computed by a cell run earlier
# in this notebook's own kernel — nothing here is re-derived or illustrative.

# Pull every method's Recall@5/MRR out of df_benchmark, plus the alpha-sweep results
scorecard = pd.DataFrame(
    {
        "Method": [
            "Semantic only",
            "Lexical (BM25) only",
            "Hybrid — RRF (k=60)",
            "Hybrid — Weighted (α=0.5)",
            f"Hybrid — Weighted (tuned α={optimal_alpha:.2f})",
            "Naive raw-score sum (no normalization)",
        ],
        "Recall@5 (5-query validation set)": [
            df_benchmark.loc[df_benchmark["Method"] == "Semantic", "Recall@5"].values[
                0
            ],
            df_benchmark.loc[
                df_benchmark["Method"] == "Lexical (BM25)", "Recall@5"
            ].values[0],
            df_benchmark.loc[
                df_benchmark["Method"] == "Hybrid (RRF)", "Recall@5"
            ].values[0],
            f"{avg_weighted_recall:.3f}",
            f"{max_recall:.3f}",
            f"{avg_naive_recall:.3f}",
        ],
        "MRR (5-query validation set)": [
            df_benchmark.loc[df_benchmark["Method"] == "Semantic", "MRR"].values[0],
            df_benchmark.loc[df_benchmark["Method"] == "Lexical (BM25)", "MRR"].values[
                0
            ],
            df_benchmark.loc[df_benchmark["Method"] == "Hybrid (RRF)", "MRR"].values[0],
            "—",
            "—",
            "—",
        ],
    }
)

print("Scorecard — every number pulled from cells already run above:")
print(scorecard.to_string(index=False))

print(f"\nHeld-out generalization check (3 new queries):")
print(f"  Tuned α={optimal_alpha:.2f}:  Recall@5 = {held_out_at_optimal:.3f}")
print(f"  Default α=0.50:  Recall@5 = {held_out_at_default:.3f}")


## What This Notebook Covered (and What It Didn't)

Section 12 of this repo's authoring guide asks every notebook to sort every technique it names into
exactly one of three tiers, so a reader never has to guess whether "mentioned" means "you'll learn
this here" or "purely for your awareness." Here's the full ledger for hybrid search:

### Tier 1 — Implemented and demonstrated (real code, real measured output)

- **Dense/embedding (bi-encoder) semantic search** — `sentence-transformers` embeddings + cosine
  similarity, measured against a rare-term failure case (Parts 1–2).
- **Sparse/lexical search (BM25)** — from-scratch IDF/TF-saturation walkthrough plus `rank_bm25`,
  measured against a synonym failure case (Parts 1, 3).
- **TF-IDF** — implemented and compared side-by-side against BM25 on a real query (Part 3).
- **Weighted linear score fusion** — implemented, normalized, and alpha-swept against a validation
  set (Parts 5, 7).
- **Reciprocal Rank Fusion (RRF)** — implemented and measured against weighted fusion and a naive
  raw-score-sum bug on the same validation set (Part 5).
- **Score normalization (min-max, z-score)** — implemented and visualized on real query scores
  (Part 6).
- **Two-stage cascade retrieval** — BM25 pre-filter + semantic rerank, implemented and run (Part 9).
- **Cross-encoder reranking** — the real `cross-encoder/ms-marco-MiniLM-L6-v2` model, run on hybrid
  RRF's own top-5 output and measured for rank changes (Part 9).
- **LangChain-style `EnsembleRetriever` production wiring** — `BM25Retriever` + FAISS + weighted RRF,
  run against three real queries (Part 8).
- **Evaluation: Recall@K and MRR** — implemented and used throughout Parts 5, 7, and 10.

### Tier 2 — Explained but not fully implemented (accurate mechanism, no full build)

- **Vector indexing — approximate (IVF-style) vs. exact search** — IVF's cluster-and-probe idea and
  HNSW's graph-walk are both explained mechanistically (Part 9), but not built, to keep this
  notebook's one hands-on infra-scaling demo scoped to cross-encoder reranking — which reuses this
  notebook's own retrieval output directly rather than introducing a new toy dataset.
- **Query expansion / rewriting** — synonym expansion, LLM-based rewriting, and pseudo-relevance
  feedback are all explained mechanistically (Part 9); none is built into a runnable pipeline here —
  doing so credibly needs either a curated synonym dictionary or a live LLM call, both a scope step
  beyond this notebook's "understand the mechanism" goal for this particular technique.
- **nDCG (graded relevance)** — formula and intuition explained in full (Part 10); not implemented
  because this notebook's validation labels are binary (relevant / not relevant), so Recall@K and MRR
  are the metrics that actually apply to the data on hand.

### Tier 3 — Named but out of scope (acknowledged, with a reason)

- **Learned fusion** (a trained ranking/fusion model over retriever outputs) — not named or built
  anywhere above this ledger; it needs labeled click-through or relevance-judgment training data and
  a trained model, which is a separate ML pipeline, not a hybrid-search mechanism itself.
- **Domain-specific embedding models** (BiomedNLP-PubMedBERT, legal-bert, CodeBERT, FinBERT) — named
  with their use cases (Part 9); swapping one in and proving it helps needs a held-out domain
  benchmark, which is a research exercise this notebook doesn't attempt.
- **Chunking strategy** — this notebook treats each of its 10 documents (and, by analogy, each of
  Riverside's chapters) as the retrieval unit; how to split a long chapter into overlapping chunks is
  a real, separate design decision that affects retrieval quality, out of scope here.
- **Metadata / filtered search** (e.g., "only search Chapter 12 onward" or "only the Fantasy
  manuscript") — not covered; this toy corpus has no queryable metadata beyond the document text
  itself, and filtering is a straightforward pre/post-filter on the same retrievers above, not a new
  retrieval mechanism.

Every technique named anywhere in this notebook sits in exactly one of the three tiers above — if a
term appears in the prose and isn't in this ledger, treat that as a bug to report, not an implied
tier 1.


## The Decision: What Does Riverside Deploy?

Riverside does **not** fine-tune changing company policies into model weights. It deploys two layers with separate responsibilities:

1. **Accepted fine-tuned assistant:** supplies the tested instruction contract and house style.
2. **Local hybrid retriever:** supplies current authorized passages from HR, Legal, Security, Finance, and editorial documents.

```text
employee question -> authorization filter -> hybrid retrieval -> cited passages -> assistant -> grounded answer
```

RRF is the default starting point because it combines ranks without pretending BM25 and cosine scores share a scale. A normalized weighted blend remains an experiment until Riverside has enough representative judgments to tune it.

### Release Conditions

- Never add raw BM25 and cosine scores.
- Version dense and sparse indexes together.
- Preserve policy ID, revision, owner, and ACL metadata through fusion.
- Decline when no authorized passage supports the answer.
- Fall back to the surviving retriever when one fails; fail closed on authorization uncertainty.
- Measure retrieval quality separately from final-answer grounding and correctness.

### Honest Limit

Ten policy passages and a small labeled question set demonstrate the mechanics, not production quality. Riverside must rerun Recall@$K$, MRR, latency, authorization, stale-version, and no-answer tests on the complete approved document collection before release.

The architecture decision is still clear: **retrieve changing company knowledge; fine-tune stable assistant behavior.**

## Production and Cloud Deployment Pattern

A production hybrid-search system separates **offline indexing** from the online query path. An ingestion job cleans and chunks source documents, attaches stable IDs plus tenant/ACL metadata, computes dense embeddings, builds the sparse BM25 index, and publishes both under one immutable index version. The online service loads that version (or points managed dense and sparse stores at it), retrieves candidates from both stores in parallel, applies tenant and authorization filters before results leave either retriever, fuses ranks with RRF, and optionally cross-encoder-reranks only the small fused candidate set.

Operate the index like a model artifact: keep a manifest with corpus/model versions, publish atomically, retain the previous version for rollback, and never mix dense and sparse artifacts from different versions. Track per-retriever latency and errors, candidate counts, overlap, empty-result rate, fallback rate, score/rank distributions, and offline Recall@K/MRR or nDCG alongside sampled relevance judgments. If one retriever or the reranker times out, return the surviving retriever's ranked results; fail closed on access-control uncertainty. The local example below uses this notebook's `SentenceTransformer`, `BM25Okapi`, cosine similarity, RRF, and optional `CrossEncoder` techniques without pretending to call a cloud vendor API.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
import hashlib
import json
import os
import shutil
import uuid


@dataclass(frozen=True)
class HybridSearchConfig:
    artifact_root: Path = Path("artifacts/hybrid-search")
    index_version: str = "v1"
    embedding_model: str = "all-MiniLM-L6-v2"
    candidate_k: int = 50
    result_k: int = 5
    rrf_k: int = 60


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def build_hybrid_artifacts(
    records: Iterable[dict[str, Any]],
    config: HybridSearchConfig,
    embedder: SentenceTransformer,
) -> Path:
    """Build and atomically publish one immutable dense+sparse index version."""
    normalized = []
    for record in records:
        normalized.append(
            {
                "id": str(record["id"]),
                "text": str(record["text"]),
                "tenant_id": str(record["tenant_id"]),
                "allowed_principals": sorted(set(record.get("allowed_principals", []))),
            }
        )
    if not normalized or len({item["id"] for item in normalized}) != len(normalized):
        raise ValueError("Records must be non-empty and have unique IDs")

    root = config.artifact_root
    root.mkdir(parents=True, exist_ok=True)
    published = root / config.index_version
    if published.exists():
        raise FileExistsError(f"Immutable index version already exists: {published}")

    staging = root / f".{config.index_version}.{uuid.uuid4().hex}.staging"
    staging.mkdir()
    try:
        texts = [item["text"] for item in normalized]
        embeddings = np.asarray(
            embedder.encode(texts, show_progress_bar=False, normalize_embeddings=True),
            dtype=np.float32,
        )
        _write_json(staging / "documents.json", normalized)
        _write_json(staging / "bm25_tokens.json", [preprocess_text(text).split() for text in texts])
        np.save(staging / "dense_embeddings.npy", embeddings, allow_pickle=False)

        artifact_names = ["documents.json", "bm25_tokens.json", "dense_embeddings.npy"]
        manifest = {
            "index_version": config.index_version,
            "embedding_model": config.embedding_model,
            "document_count": len(normalized),
            "embedding_dimensions": int(embeddings.shape[1]),
            "sha256": {name: _sha256(staging / name) for name in artifact_names},
        }
        _write_json(staging / "manifest.json", manifest)
        os.replace(staging, published)
    except Exception:
        shutil.rmtree(staging, ignore_errors=True)
        raise

    pointer_tmp = root / f".CURRENT.{uuid.uuid4().hex}"
    pointer_tmp.write_text(config.index_version, encoding="utf-8")
    os.replace(pointer_tmp, root / "CURRENT")
    return published

In [ ]:
import logging
import time

logger = logging.getLogger("hybrid_search")


@dataclass
class HybridArtifacts:
    documents: list[dict[str, Any]]
    embeddings: np.ndarray
    bm25: BM25Okapi
    manifest: dict[str, Any]


def load_hybrid_artifacts(
    config: HybridSearchConfig, version: str | None = None
) -> HybridArtifacts:
    """Load one complete index version and reject partial or corrupted artifacts."""
    selected_version = version or (config.artifact_root / "CURRENT").read_text(encoding="utf-8").strip()
    index_dir = config.artifact_root / selected_version
    manifest = json.loads((index_dir / "manifest.json").read_text(encoding="utf-8"))

    for name, expected_hash in manifest["sha256"].items():
        if _sha256(index_dir / name) != expected_hash:
            raise ValueError(f"Checksum mismatch for {name} in index {selected_version}")
    if manifest["embedding_model"] != config.embedding_model:
        raise ValueError("Configured embedding model does not match the published index")

    loaded_documents = json.loads((index_dir / "documents.json").read_text(encoding="utf-8"))
    bm25_tokens = json.loads((index_dir / "bm25_tokens.json").read_text(encoding="utf-8"))
    loaded_embeddings = np.load(
        index_dir / "dense_embeddings.npy", mmap_mode="r", allow_pickle=False
    )
    expected_count = manifest["document_count"]
    if len(loaded_documents) != expected_count or loaded_embeddings.shape[0] != expected_count:
        raise ValueError("Index artifacts have inconsistent document counts")

    return HybridArtifacts(
        documents=loaded_documents,
        embeddings=loaded_embeddings,
        bm25=BM25Okapi(bm25_tokens),
        manifest=manifest,
    )


def _authorized_indices(
    records: list[dict[str, Any]], tenant_id: str, principals: set[str]
) -> list[int]:
    return [
        index
        for index, record in enumerate(records)
        if record["tenant_id"] == tenant_id
        and (
            not record["allowed_principals"]
            or bool(principals.intersection(record["allowed_principals"]))
        )
    ]


def _top_authorized(scores: np.ndarray, allowed: list[int], top_k: int) -> list[tuple[int, float]]:
    ranked = sorted(allowed, key=lambda index: float(scores[index]), reverse=True)
    return [(index, float(scores[index])) for index in ranked[:top_k]]


def search_hybrid_index(
    query: str,
    artifacts: HybridArtifacts,
    embedder: SentenceTransformer,
    config: HybridSearchConfig,
    tenant_id: str,
    principals: set[str],
    reranker: Any | None = None,
) -> dict[str, Any]:
    """Retrieve, fuse, optionally rerank, and expose compact operational telemetry."""
    allowed = _authorized_indices(artifacts.documents, tenant_id, principals)
    if not allowed:
        return {"results": [], "telemetry": {"authorized_candidates": 0, "fallbacks": []}}

    timings_ms: dict[str, float] = {}
    fallbacks: list[str] = []
    dense_hits: list[tuple[int, float]] = []
    sparse_hits: list[tuple[int, float]] = []

    started = time.perf_counter()
    try:
        query_vector = embedder.encode(
            [query], show_progress_bar=False, normalize_embeddings=True
        )
        dense_scores = cosine_similarity(query_vector, artifacts.embeddings)[0]
        dense_hits = _top_authorized(dense_scores, allowed, config.candidate_k)
    except Exception as error:
        logger.warning("Dense retrieval unavailable; using sparse fallback: %s", error)
        fallbacks.append("dense")
    timings_ms["dense"] = (time.perf_counter() - started) * 1000

    started = time.perf_counter()
    try:
        sparse_scores = artifacts.bm25.get_scores(preprocess_text(query).split())
        sparse_hits = _top_authorized(sparse_scores, allowed, config.candidate_k)
    except Exception as error:
        logger.warning("Sparse retrieval unavailable; using dense fallback: %s", error)
        fallbacks.append("sparse")
    timings_ms["sparse"] = (time.perf_counter() - started) * 1000

    if dense_hits and sparse_hits:
        candidates = reciprocal_rank_fusion(
            dense_hits, sparse_hits, k=config.rrf_k
        )[: config.candidate_k]
        retrieval_mode = "hybrid_rrf"
    elif dense_hits or sparse_hits:
        candidates = (dense_hits or sparse_hits)[: config.candidate_k]
        retrieval_mode = "dense_only" if dense_hits else "sparse_only"
    else:
        raise RuntimeError("Both dense and sparse retrieval failed")

    if reranker is not None and candidates:
        started = time.perf_counter()
        try:
            pairs = [(query, artifacts.documents[index]["text"]) for index, _ in candidates]
            rerank_scores = reranker.predict(pairs)
            candidates = sorted(
                zip([index for index, _ in candidates], rerank_scores),
                key=lambda item: float(item[1]),
                reverse=True,
            )
            timings_ms["rerank"] = (time.perf_counter() - started) * 1000
        except Exception as error:
            logger.warning("Reranker unavailable; returning fused order: %s", error)
            fallbacks.append("reranker")

    results = [
        {**artifacts.documents[index], "score": float(score), "retrieval_mode": retrieval_mode}
        for index, score in candidates[: config.result_k]
    ]
    return {
        "results": results,
        "telemetry": {
            "index_version": artifacts.manifest["index_version"],
            "authorized_candidates": len(allowed),
            "dense_candidates": len(dense_hits),
            "sparse_candidates": len(sparse_hits),
            "timings_ms": timings_ms,
            "fallbacks": fallbacks,
        },
    }

In [ ]:
RUN_PRODUCTION_INDEXING = False
RUN_PRODUCTION_QUERY = False
RUN_PRODUCTION_RERANKING = False

production_config = HybridSearchConfig(
    artifact_root=Path("artifacts/hybrid-search"),
    index_version="riverside-policies-v1",
    candidate_k=50,
    result_k=5,
)

production_embedder = None
if RUN_PRODUCTION_INDEXING or RUN_PRODUCTION_QUERY:
    production_embedder = SentenceTransformer(production_config.embedding_model)

if RUN_PRODUCTION_INDEXING:
    production_records = [
        {
            "id": f"riverside-policy-{index}",
            "text": text,
            "tenant_id": "riverside-demo",
            "allowed_principals": ["policy-readers"],
        }
        for index, text in enumerate(documents)
    ]
    published_path = build_hybrid_artifacts(
        production_records, production_config, production_embedder
    )
    print(f"Published immutable hybrid index: {published_path}")

if RUN_PRODUCTION_QUERY:
    production_artifacts = load_hybrid_artifacts(production_config)
    production_reranker = (
        CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
        if RUN_PRODUCTION_RERANKING
        else None
    )
    response = search_hybrid_index(
        query="can staff paste draft novels into ChatGPT",
        artifacts=production_artifacts,
        embedder=production_embedder,
        config=production_config,
        tenant_id="riverside-demo",
        principals={"policy-readers"},
        reranker=production_reranker,
    )
    print(json.dumps(response, indent=2))
else:
    print("Production indexing/query examples are disabled; set RUN_PRODUCTION_* explicitly to run.")